<a href="https://colab.research.google.com/github/Nityahapani/aji-river-pollution-mapper/blob/main/Hedge_Fund_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install yfinance pandas numpy requests beautifulsoup4 fredapi

In [2]:
import os

dirs = [
    "/content/hedge_fund/scripts",
    "/content/hedge_fund/data/raw/prices",
    "/content/hedge_fund/data/processed/prices",
    "/content/hedge_fund/universe",
]

for d in dirs:
    os.makedirs(d, exist_ok=True)

print("✓ Folder structure created")

✓ Folder structure created


In [3]:
import shutil, os

scripts = [
    "universe.py",
    "price_data.py",
    "rbi_data.py",
    "reconstitution_data.py",
    "data_master.py"
]

for script in scripts:
    src = f"/content/{script}"
    dst = f"/content/hedge_fund/scripts/{script}"
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"✓ {script}")
    else:
        print(f"✗ Missing: {script}")

✓ universe.py
✓ price_data.py
✓ rbi_data.py
✓ reconstitution_data.py
✓ data_master.py


In [4]:
import sys, os
os.chdir("/content/hedge_fund")
sys.path.insert(0, "/content/hedge_fund/scripts")

from data_master import main
main()

ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['ABLBL.NS']: YFPricesMissingError('possibly delisted; no price data found  (1d 2014-01-01 -> 2024-12-31) (Yahoo error = "Data doesn\'t exist for startDate = 1388514600, endDate = 1735583400")')
ERROR:yfinance:
2 Failed downloads:
ERROR:yfinance:['AEGISVOPAK.NS', 'AGARWALEYE.NS']: YFPricesMissingError('possibly delisted; no price data found  (1d 2014-01-01 -> 2024-12-31) (Yahoo error = "Data doesn\'t exist for startDate = 1388514600, endDate = 1735583400")')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['ATHERENERG.NS']: YFPricesMissingError('possibly delisted; no price data found  (1d 2014-01-01 -> 2024-12-31) (Yahoo error = "Data doesn\'t exist for startDate = 1388514600, endDate = 1735583400")')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['COHANCE.NS']: YFPricesMissingError('possibly delisted; no price data found  (1d 2014-01-01 -> 2024-12-31) (Yahoo error = "Data doesn\'t exist for startDate = 1388514600, endDate = 17355



  HEDGE FUND DATA PIPELINE — QUALITY REPORT
  Generated: 2026-03-05T03:16:37
  Status:    SUCCESS

PIPELINE STEPS
-----------------------------------------------------------------
  ✓  Universe Construction                    2.39s
  ✓  Price Data Pipeline                      197.3s
  ✓  RBI Rate Data Pipeline                   1.67s
  ✓  Reconstitution Data Pipeline             0.28s

FILE VALIDATION
-----------------------------------------------------------------
  ✓  universe_monthly.csv                         4105.6 KB  [OK]
  ✓  universe_metadata.csv                          45.4 KB  [OK]
  ✓  survivorship_log.csv                           14.0 KB  [OK]
  ✓  universe_stats.json                             0.6 KB  [OK]
  ✓  close.csv                                   19268.4 KB  [OK]
  ✓  returns.csv                                 21884.5 KB  [OK]
  ✓  volume.csv                                  10172.8 KB  [OK]
  ✓  price_quality_report.csv                       28.7 KB  [OK

{'generated_at': '2026-03-05T03:16:37.950839',
 'pipeline_status': 'SUCCESS',
 'steps': [{'step': 'Universe Construction',
   'module': 'universe.py',
   'status': 'SUCCESS',
   'start_time': '2026-03-05T03:13:16.281653',
   'end_time': '2026-03-05T03:13:18.670560',
   'duration_s': 2.39,
   'error': None,
   'outputs': []},
  {'step': 'Price Data Pipeline',
   'module': 'price_data.py',
   'status': 'SUCCESS',
   'start_time': '2026-03-05T03:13:18.676344',
   'end_time': '2026-03-05T03:16:35.978956',
   'duration_s': 197.3,
   'error': None,
   'outputs': []},
  {'step': 'RBI Rate Data Pipeline',
   'module': 'rbi_data.py',
   'status': 'SUCCESS',
   'start_time': '2026-03-05T03:16:36.001472',
   'end_time': '2026-03-05T03:16:37.667655',
   'duration_s': 1.67,
   'error': None,
   'outputs': []},
  {'step': 'Reconstitution Data Pipeline',
   'module': 'reconstitution_data.py',
   'status': 'SUCCESS',
   'start_time': '2026-03-05T03:16:37.671260',
   'end_time': '2026-03-05T03:16:37.95

In [5]:
import pandas as pd
import os

bench = pd.read_csv("/content/hedge_fund/data/processed/benchmark_prices.csv", index_col=0)
bench_returns = bench.pct_change().dropna()
bench_returns.to_csv("/content/hedge_fund/data/processed/benchmark_returns.csv")
print(f"✓ benchmark_returns.csv created — {bench_returns.shape}")

✓ benchmark_returns.csv created — (2869, 2)


In [6]:
import yfinance as yf
import pandas as pd

nifty50  = yf.download("^NSEI",   start="2014-01-01", end="2024-12-31",
                        auto_adjust=True, progress=False)
nifty500 = yf.download("^CRSLDX", start="2014-01-01", end="2024-12-31",
                        auto_adjust=True, progress=False)

bench = pd.DataFrame({
    "NIFTY50":  nifty50["Close"].squeeze(),
    "NIFTY500": nifty500["Close"].squeeze()
})
bench.index = pd.to_datetime(bench.index).tz_localize(None)
bench.index.name = "date"
bench.to_csv("/content/hedge_fund/data/processed/benchmark_prices.csv")

bench_returns = bench.pct_change().dropna()
bench_returns.to_csv("/content/hedge_fund/data/processed/benchmark_returns.csv")

print(f"✓ Benchmark prices: {bench.shape}")
print(f"✓ Benchmark returns: {bench_returns.shape}")
print(bench.tail(3))

✓ Benchmark prices: (2706, 2)
✓ Benchmark returns: (2704, 2)
                 NIFTY50      NIFTY500
date                                  
2024-12-26  23750.199219  22430.349609
2024-12-27  23813.400391  22445.199219
2024-12-30  23644.900391  22357.150391


In [7]:
import shutil
from google.colab import files

shutil.make_archive("/content/hedge_fund_data", "zip",
                    "/content/hedge_fund/data/processed")
files.download("/content/hedge_fund_data.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
import sys, os

# Set the correct base directory explicitly
os.chdir("/content/hedge_fund")

# Fix paths by setting an environment variable all scripts will pick up
os.environ["HEDGE_FUND_BASE"] = "/content/hedge_fund"

# Add correct paths
sys.path.insert(0, "/content/hedge_fund/scripts/signals")
sys.path.insert(0, "/content/hedge_fund/scripts")

# Verify the data files exist before running
files_to_check = [
    "/content/hedge_fund/data/processed/close.csv",
    "/content/hedge_fund/data/processed/rate_signal.csv",
    "/content/hedge_fund/data/processed/reconstitution_events.csv",
]

all_good = True
for f in files_to_check:
    if os.path.exists(f):
        print(f"✓ Found: {f}")
    else:
        print(f"✗ Missing: {f}")
        all_good = False

if all_good:
    print("\n✓ All data files found — ready to run signals")
else:
    print("\n✗ Some files missing — did you unzip hedge_fund_data.zip?")

✓ Found: /content/hedge_fund/data/processed/close.csv
✓ Found: /content/hedge_fund/data/processed/rate_signal.csv
✓ Found: /content/hedge_fund/data/processed/reconstitution_events.csv

✓ All data files found — ready to run signals


In [12]:
import os, sys

# Move all signal scripts to the correct location
os.makedirs("/content/hedge_fund/scripts/signals", exist_ok=True)

scripts = ["signal_master.py", "momentum_signal.py",
           "rate_signal_engine.py", "recon_signal_engine.py"]

for script in scripts:
    src = f"/content/{script}"
    dst = f"/content/hedge_fund/scripts/signals/{script}"
    if os.path.exists(src):
        import shutil
        shutil.copy(src, dst)
        print(f"✓ Moved {script}")
    elif os.path.exists(dst):
        print(f"✓ Already in place: {script}")
    else:
        print(f"✗ Not found: {script}")

✓ Moved signal_master.py
✓ Moved momentum_signal.py
✓ Moved rate_signal_engine.py
✓ Moved recon_signal_engine.py


In [14]:
import os, sys

# Move all signal scripts to the correct location
os.makedirs("/content/hedge_fund/scripts/signals", exist_ok=True)

scripts = ["signal_master.py", "momentum_signal.py",
           "rate_signal_engine.py", "recon_signal_engine.py"]

for script in scripts:
    src = f"/content/{script}"
    dst = f"/content/hedge_fund/scripts/signals/{script}"
    if os.path.exists(src):
        import shutil
        shutil.copy(src, dst)
        print(f"✓ Moved {script}")
    elif os.path.exists(dst):
        print(f"✓ Already in place: {script}")
    else:
        print(f"✗ Not found: {script}")

✓ Moved signal_master.py
✓ Moved momentum_signal.py
✓ Moved rate_signal_engine.py
✓ Moved recon_signal_engine.py


In [16]:
import os, sys, importlib, runpy

# Clear everything
for mod in list(sys.modules.keys()):
    if any(x in mod for x in ["signal", "momentum", "rate", "recon"]):
        del sys.modules[mod]

# Force correct working directory
os.chdir("/content/hedge_fund")

# Remove /content/ from path so old scripts aren't found
sys.path = [p for p in sys.path if p != "/content"]

# Add correct paths at the front
sys.path.insert(0, "/content/hedge_fund/scripts/signals")
sys.path.insert(0, "/content/hedge_fund/scripts")

# Verify the script is loading from the right place
import momentum_signal
print(f"Loading from: {momentum_signal.__file__}")
print(f"BASE_DIR resolves to: {momentum_signal.BASE_DIR}")

ModuleNotFoundError: No module named 'momentum_signal'

In [20]:
import runpy, sys, os

os.chdir("/content/hedge_fund")
SIGNALS_DIR = "/content/hedge_fund/scripts/signals"

# Ensure path is set inside each script's execution
sys.path.insert(0, SIGNALS_DIR)
sys.path.insert(0, "/content/hedge_fund/scripts")

print("Running Momentum Signal Engine...")
runpy.run_path(f"{SIGNALS_DIR}/momentum_signal.py", run_name="__main__")
print("\n✓ Momentum done")

print("\nRunning Rate Sensitivity Signal Engine...")
runpy.run_path(f"{SIGNALS_DIR}/rate_signal_engine.py", run_name="__main__")
print("\n✓ Rate Sensitivity done")

print("\nRunning Reconstitution Signal Engine...")
runpy.run_path(f"{SIGNALS_DIR}/recon_signal_engine.py", run_name="__main__")
print("\n✓ Reconstitution done")

print("\n✓ All 3 signal engines complete")

Running Momentum Signal Engine...

✓ Momentum done

Running Rate Sensitivity Signal Engine...

✓ Rate Sensitivity done

Running Reconstitution Signal Engine...

✓ Reconstitution done

✓ All 3 signal engines complete


In [21]:
import os

signal_dir = "/content/hedge_fund/data/processed/signals"
files = os.listdir(signal_dir)

total_size = 0
for f in sorted(files):
    size = os.path.getsize(os.path.join(signal_dir, f)) / 1024
    total_size += size
    print(f"✓  {f:<45} {size:>8.1f} KB")

print(f"\nTotal: {len(files)} files, {total_size/1024:.1f} MB")

✓  momentum_monthly.csv                            2736.2 KB
✓  momentum_signal.csv                             7085.0 KB
✓  rate_sensitivity_map.csv                          19.5 KB
✓  rate_signal.csv                                 9610.4 KB
✓  rate_signal_detailed.csv                       61595.1 KB
✓  rate_signal_stats.json                             0.9 KB
✓  recon_concurrent_positions.csv                     5.4 KB
✓  recon_signal.csv                                 509.3 KB
✓  recon_signal_detailed.csv                         64.5 KB
✓  recon_signal_stats.json                            0.4 KB
✓  signal_summary.txt                                 1.6 KB

Total: 11 files, 79.7 MB


In [23]:
import shutil
from google.colab import files

shutil.make_archive("/content/hedge_fund_signals", "zip",
                    "/content/hedge_fund/data/processed/signals")
files.download("/content/hedge_fund_signals.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [29]:
import os
os.makedirs("/content/hedge_fund/scripts/portfolio", exist_ok=True)

code = open("/dev/stdin").read() if False else None

# Write the script directly
import urllib.request
# Since we can't fetch from GitHub, write it inline
print("Creating portfolio_construction.py...")

# Check if it was uploaded to /content first
if os.path.exists("/content/portfolio_construction.py"):
    import shutil
    shutil.copy("/content/portfolio_construction.py",
                "/content/hedge_fund/scripts/portfolio/portfolio_construction.py")
    print("✓ Moved from /content/")
else:
    print("✗ File not found in /content/ — please upload it first")
    print("  1. Click the folder icon in the left sidebar")
    print("  2. Click the upload icon")
    print("  3. Upload portfolio_construction.py")
    print("  4. Then re-run this cell")

Creating portfolio_construction.py...
✓ Moved from /content/


In [30]:
import runpy, os
os.chdir("/content/hedge_fund")
runpy.run_path(
    "/content/hedge_fund/scripts/portfolio/portfolio_construction.py",
    run_name="__main__"
)

{'__name__': '__main__',
 '__doc__': '\nportfolio_construction.py\n=========================\nPortfolio Construction Model — combines signals from all three sleeves\ninto daily portfolio weights with risk constraints.\n\nSleeve allocations:\n    - Sleeve A (Momentum):          50% of gross exposure\n    - Sleeve B (Rate Sensitivity):  30% of gross exposure\n    - Sleeve C (Reconstitution):    20% of gross exposure\n\nConstraints enforced:\n    - Max single position:    5% of NAV (long or short)\n    - Max sector exposure:    25% of NAV per sector\n    - Gross exposure:         80% - 150% of NAV\n    - Net exposure:           -20% to +40% of NAV\n    - Volatility target:      12% annualised (scale portfolio to hit this)\n    - Min position size:      0.5% (below this, set to zero — avoid tiny positions)\n\nConflict resolution (stock appears in multiple sleeves):\n    - Take weighted average of signals, weighted by sleeve allocation\n    - If signals conflict (one says LONG, one says SHO

In [34]:
result["main"]()

(            INDIGO.NS  CAIRN.NS  PCJEWELLER.NS  BHEL.NS   GPIL.NS  \
 date                                                                
 2015-01-07       0.05      0.05            0.0      0.0  0.000000   
 2015-01-08       0.05      0.05            0.0      0.0  0.000000   
 2015-01-09       0.05      0.05            0.0      0.0  0.000000   
 2015-01-12       0.05      0.05            0.0      0.0  0.000000   
 2015-01-13       0.05      0.05            0.0      0.0  0.000000   
 ...               ...       ...            ...      ...       ...   
 2023-09-06       0.00      0.00            0.0      0.0  0.005069   
 2023-09-07       0.00      0.00            0.0      0.0  0.005065   
 2023-09-08       0.00      0.00            0.0      0.0  0.005066   
 2023-09-11       0.00      0.00            0.0      0.0  0.005067   
 2023-09-12       0.00      0.00            0.0      0.0  0.005068   
 
             USHAMART.NS  SUZLON.NS  CANFINHOME.NS  BEML.NS  GVT&D.NS  ...  \
 date     

In [35]:
import shutil
from google.colab import files

shutil.make_archive("/content/hedge_fund_portfolio", "zip",
                    "/content/hedge_fund/data/processed/portfolio")
files.download("/content/hedge_fund_portfolio.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [36]:
import pandas as pd

weights = pd.read_csv("/content/hedge_fund/data/processed/portfolio/weights_daily.csv",
                      index_col=0, parse_dates=True)
exposure = pd.read_csv("/content/hedge_fund/data/processed/portfolio/exposure_daily.csv",
                       index_col=0, parse_dates=True)

# Filter to active days only
active = exposure[exposure["n_positions"] > 0]

print(f"Total days:          {len(exposure)}")
print(f"Active trading days: {len(active)}")
print(f"Avg positions:       {active['n_positions'].mean():.1f}")
print(f"Avg gross exposure:  {active['gross'].mean():.1%}")
print(f"Avg net exposure:    {active['net'].mean():.1%}")
print(f"Avg longs:           {active['n_longs'].mean():.1f}")
print(f"Avg shorts:          {active['n_shorts'].mean():.1f}")
print(f"\nExposure over time:")
print(active[["gross", "net", "n_positions"]].resample("YE").mean().round(3))

Total days:          2608
Active trading days: 226
Avg positions:       12.8
Avg gross exposure:  8.0%
Avg net exposure:    8.0%
Avg longs:           12.8
Avg shorts:          0.0

Exposure over time:
            gross    net  n_positions
date                                 
2015-12-31  0.087  0.087        1.739
2016-12-31  0.085  0.085       14.480
2017-12-31  0.022  0.022        4.143
2018-12-31    NaN    NaN          NaN
2019-12-31  0.005  0.005        1.000
2020-12-31  0.008  0.008        1.500
2021-12-31    NaN    NaN          NaN
2022-12-31    NaN    NaN          NaN
2023-12-31  0.109  0.109       20.063


In [37]:
import pandas as pd
import numpy as np
import os, json

os.chdir("/content/hedge_fund")
BASE = "/content/hedge_fund/data/processed"

# ── Load signals directly ──────────────────────────────────────────────────────
print("Loading signals...")
mom   = pd.read_csv(f"{BASE}/signals/momentum_signal.csv",   index_col=0, parse_dates=True)
rate  = pd.read_csv(f"{BASE}/signals/rate_signal.csv",       index_col=0, parse_dates=True)
recon = pd.read_csv(f"{BASE}/signals/recon_signal.csv",      index_col=0, parse_dates=True)
rets  = pd.read_csv(f"{BASE}/returns.csv",                   index_col=0, parse_dates=True)
meta  = pd.read_csv(f"{BASE}/universe_metadata.csv")

print(f"Momentum:  {mom.shape}")
print(f"Rate:      {rate.shape}")
print(f"Recon:     {recon.shape}")
print(f"Returns:   {rets.shape}")

# Quick signal check — are there non-zero values?
print(f"\nMomentum non-zero values: {(mom != 0).sum().sum():,}")
print(f"Rate non-zero values:     {(rate != 0).sum().sum():,}")
print(f"Recon non-zero values:    {(recon != 0).sum().sum():,}")

# Show sample of momentum signal
print(f"\nMomentum sample (2020-06):")
sample = mom.loc["2020-06-01":"2020-06-05"]
print(sample.iloc[:, :5])

Loading signals...
Momentum:  (2608, 461)
Rate:      (2609, 341)
Recon:     (2609, 47)
Returns:   (2869, 461)

Momentum non-zero values: 1,201,220
Rate non-zero values:     485,586
Recon non-zero values:    705

Momentum sample (2020-06):
            360ONE.NS  3MINDIA.NS  AARTIIND.NS  AAVAS.NS  ABB.NS
date                                                            
2020-06-01        NaN      0.2752       0.6948    0.3678  0.1499
2020-06-02        NaN      0.2752       0.6948    0.3678  0.1499
2020-06-03        NaN      0.2752       0.6948    0.3678  0.1499
2020-06-04        NaN      0.2752       0.6948    0.3678  0.1499
2020-06-05        NaN      0.2752       0.6948    0.3678  0.1499


In [38]:
import pandas as pd
import numpy as np
import os, json

BASE     = "/content/hedge_fund/data/processed"
PORT_DIR = f"{BASE}/portfolio"
os.makedirs(PORT_DIR, exist_ok=True)

# ── Config ─────────────────────────────────────────────────────────────────────
SLEEVE_W       = {"momentum": 0.50, "rate": 0.30, "recon": 0.20}
MAX_POS        = 0.05
MIN_POS        = 0.005
MAX_SECTOR     = 0.25
MAX_GROSS      = 1.50
MAX_NET        = 0.40
MIN_NET        = -0.20
VOL_TARGET     = 0.12
VOL_LOOKBACK   = 63

# ── Load data ──────────────────────────────────────────────────────────────────
print("Loading data...")
mom   = pd.read_csv(f"{BASE}/signals/momentum_signal.csv",  index_col=0, parse_dates=True).fillna(0)
rate  = pd.read_csv(f"{BASE}/signals/rate_signal.csv",      index_col=0, parse_dates=True).fillna(0)
recon = pd.read_csv(f"{BASE}/signals/recon_signal.csv",     index_col=0, parse_dates=True).fillna(0)
rets  = pd.read_csv(f"{BASE}/returns.csv",                  index_col=0, parse_dates=True).fillna(0)
meta  = pd.read_csv(f"{BASE}/universe_metadata.csv")

sector_map = {}
for _, r in meta.iterrows():
    sector_map[r["symbol"]]          = r["sector"]
    sector_map[r["symbol"] + ".NS"]  = r["sector"]

SECTOR_SENSITIVITY = {
    "Banking": 1.0, "NBFC & Financials": 0.9, "Real Estate": 0.85,
    "Utilities": 0.7, "Auto": 0.6, "Infrastructure": 0.4,
    "Consumer": 0.2, "Commodities": -0.1, "Energy": -0.2,
    "Telecom": -0.2, "Media": -0.3, "Chemicals": -0.4,
    "Healthcare": -0.6, "Technology": -0.8, "Other": 0.0,
}

print(f"Momentum: {mom.shape} | Rate: {rate.shape} | Recon: {recon.shape}")

# ── Get common trading days ────────────────────────────────────────────────────
trading_days = mom.index[(mom.index >= "2015-01-01") & (mom.index <= "2024-12-31")]
print(f"Trading days: {len(trading_days)}")

# ── Build portfolio day by day ─────────────────────────────────────────────────
all_weights  = []
exposure_log = []

for i, date in enumerate(trading_days):

    # Get signals for this date
    m_sig = mom.loc[date]   if date in mom.index   else pd.Series(dtype=float)
    r_sig = rate.loc[date]  if date in rate.index  else pd.Series(dtype=float)
    rc_sig = recon.loc[date] if date in recon.index else pd.Series(dtype=float)

    # Align to common universe
    all_stocks = sorted(set(m_sig.index) | set(r_sig.index) | set(rc_sig.index))
    m_sig  = m_sig.reindex(all_stocks).fillna(0)
    r_sig  = r_sig.reindex(all_stocks).fillna(0)
    rc_sig = rc_sig.reindex(all_stocks).fillna(0)

    # Weighted combination
    combined = (SLEEVE_W["momentum"] * m_sig +
                SLEEVE_W["rate"]     * r_sig +
                SLEEVE_W["recon"]    * rc_sig)

    # Keep only stocks with meaningful signal
    combined = combined[combined.abs() > 0.01]

    if len(combined) < 5:
        exposure_log.append({"date": date, "gross": 0, "net": 0,
                             "long": 0, "short": 0, "n_positions": 0,
                             "n_longs": 0, "n_shorts": 0, "vol_scale": 1.0})
        continue

    # Separate longs and shorts
    longs  = combined[combined > 0]
    shorts = combined[combined < 0]

    weights = pd.Series(dtype=float)
    if len(longs) > 0:
        w_long  = longs / longs.sum() * 0.50
        weights = pd.concat([weights, w_long])
    if len(shorts) > 0:
        w_short = shorts / shorts.abs().sum() * (-0.50)
        weights = pd.concat([weights, w_short])

    # Position cap
    weights = weights.clip(-MAX_POS, MAX_POS)

    # Sector cap
    for sector in set(sector_map.get(s, "Other") for s in weights.index):
        sec_stocks = [s for s in weights.index if sector_map.get(s, "Other") == sector]
        sec_exp    = weights[sec_stocks].abs().sum()
        if sec_exp > MAX_SECTOR:
            scale = MAX_SECTOR / sec_exp
            weights[sec_stocks] *= scale

    # Vol targeting
    common_stocks = [s for s in weights.index if s in rets.columns]
    if len(common_stocks) >= 5:
        idx       = rets.index.get_loc(date) if date in rets.index else -1
        if idx > VOL_LOOKBACK:
            ret_win  = rets.iloc[idx - VOL_LOOKBACK:idx][common_stocks].fillna(0)
            w_arr    = weights[common_stocks].values
            port_ret = ret_win.values @ w_arr
            port_vol = np.std(port_ret) * np.sqrt(252)
            if port_vol > 0.001:
                scale = np.clip(VOL_TARGET / port_vol, 0.5, 2.0)
                weights = weights * scale

    # Net/gross exposure limits
    gross = weights.abs().sum()
    net   = weights.sum()
    if gross > MAX_GROSS:
        weights *= MAX_GROSS / gross
    if net > MAX_NET:
        weights[weights > 0] -= (net - MAX_NET) / max((weights > 0).sum(), 1)
    if net < MIN_NET:
        weights[weights < 0] += (MIN_NET - net) / max((weights < 0).sum(), 1)

    # Zero out tiny positions
    weights[weights.abs() < MIN_POS] = 0
    weights = weights[weights != 0]

    gross = weights.abs().sum()
    net   = weights.sum()

    weights.name = date
    all_weights.append(weights)

    exposure_log.append({
        "date":        date,
        "gross":       round(gross, 4),
        "net":         round(net, 4),
        "long":        round(weights[weights > 0].sum(), 4),
        "short":       round(weights[weights < 0].sum(), 4),
        "n_positions": len(weights),
        "n_longs":     int((weights > 0).sum()),
        "n_shorts":    int((weights < 0).sum()),
        "vol_scale":   round(float(np.clip(VOL_TARGET / (port_vol if 'port_vol' in dir() else 0.12), 0.5, 2.0)), 4),
    })

    if i % 500 == 0:
        print(f"  {date.date()}: {len(weights)} positions | gross={gross:.1%} net={net:.1%}")

# ── Assemble and save ──────────────────────────────────────────────────────────
print("\nAssembling matrices...")
weights_df  = pd.DataFrame(all_weights).fillna(0)
exposure_df = pd.DataFrame(exposure_log).set_index("date")

weights_df.to_csv(f"{PORT_DIR}/weights_daily.csv")
weights_df.resample("ME").last().to_csv(f"{PORT_DIR}/weights_monthly.csv")
exposure_df.to_csv(f"{PORT_DIR}/exposure_daily.csv")

# ── Summary ────────────────────────────────────────────────────────────────────
active = exposure_df[exposure_df["n_positions"] > 0]
print(f"\n✓ Portfolio construction complete")
print(f"  Total days:          {len(exposure_df)}")
print(f"  Active trading days: {len(active)}")
print(f"  Avg positions:       {active['n_positions'].mean():.1f}")
print(f"  Avg gross exposure:  {active['gross'].mean():.1%}")
print(f"  Avg net exposure:    {active['net'].mean():.1%}")
print(f"  Avg longs:           {active['n_longs'].mean():.1f}")
print(f"  Avg shorts:          {active['n_shorts'].mean():.1f}")
print(f"\nAnnual exposure summary:")
print(active[["gross","net","n_positions"]].resample("YE").mean().round(3))

Loading data...
Momentum: (2608, 461) | Rate: (2609, 341) | Recon: (2609, 47)
Trading days: 2608
  2016-12-01: 0 positions | gross=0.0% net=0.0%
  2018-11-01: 0 positions | gross=0.0% net=0.0%
  2020-10-01: 0 positions | gross=0.0% net=0.0%
  2022-09-01: 0 positions | gross=0.0% net=0.0%
  2024-08-01: 0 positions | gross=0.0% net=0.0%

Assembling matrices...

✓ Portfolio construction complete
  Total days:          2608
  Active trading days: 199
  Avg positions:       12.4
  Avg gross exposure:  7.0%
  Avg net exposure:    7.0%
  Avg longs:           12.4
  Avg shorts:          0.0

Annual exposure summary:
            gross    net  n_positions
date                                 
2016-12-31  0.082  0.082       14.150
2017-12-31  0.021  0.021        4.030
2018-12-31    NaN    NaN          NaN
2019-12-31    NaN    NaN          NaN
2020-12-31  0.005  0.005        1.000
2021-12-31    NaN    NaN          NaN
2022-12-31    NaN    NaN          NaN
2023-12-31  0.078  0.078       14.603


In [39]:
import pandas as pd
import numpy as np

BASE = "/content/hedge_fund/data/processed"

mom  = pd.read_csv(f"{BASE}/signals/momentum_signal.csv",  index_col=0, parse_dates=True)
rate = pd.read_csv(f"{BASE}/signals/rate_signal.csv",      index_col=0, parse_dates=True)
recon = pd.read_csv(f"{BASE}/signals/recon_signal.csv",    index_col=0, parse_dates=True)

# Check signal distributions
print("=== MOMENTUM SIGNAL ===")
mom_vals = mom.values.flatten()
mom_vals = mom_vals[~np.isnan(mom_vals)]
print(f"Min: {mom_vals.min():.4f} | Max: {mom_vals.max():.4f} | Mean: {mom_vals.mean():.4f}")
print(f"Negative values: {(mom_vals < 0).sum():,} / {len(mom_vals):,}")
print(f"Values > 0.01:   {(mom_vals > 0.01).sum():,}")
print(f"Values < -0.01:  {(mom_vals < -0.01).sum():,}")

print("\n=== RATE SIGNAL ===")
rate_vals = rate.values.flatten()
rate_vals = rate_vals[~np.isnan(rate_vals)]
print(f"Min: {rate_vals.min():.4f} | Max: {rate_vals.max():.4f} | Mean: {rate_vals.mean():.4f}")
print(f"Negative values: {(rate_vals < 0).sum():,} / {len(rate_vals):,}")

print("\n=== RECON SIGNAL ===")
recon_vals = recon.values.flatten()
recon_vals = recon_vals[~np.isnan(recon_vals)]
print(f"Min: {recon_vals.min():.4f} | Max: {recon_vals.max():.4f} | Mean: {recon_vals.mean():.4f}")
print(f"Negative values: {(recon_vals < 0).sum():,} / {len(recon_vals):,}")

# Check a specific date with known signals
print("\n=== SAMPLE DATE: 2022-06-01 (hiking cycle — should have shorts) ===")
sample = mom.loc["2022-06-01"]
print(f"Non-zero: {(sample != 0).sum()} | Negative: {(sample < 0).sum()} | Positive: {(sample > 0).sum()}")
print(f"Min value: {sample.min():.4f} | Max value: {sample.max():.4f}")
print("\nBottom 5 stocks (should be shorts):")
print(sample.nsmallest(5))

=== MOMENTUM SIGNAL ===
Min: -0.9955 | Max: 1.0000 | Mean: 0.0138
Negative values: 467,962 / 950,378
Values > 0.01:   477,044
Values < -0.01:  463,658

=== RATE SIGNAL ===
Min: -0.9000 | Max: 0.9000 | Mean: 0.0068
Negative values: 242,205 / 889,669

=== RECON SIGNAL ===
Min: -0.1500 | Max: 0.1500 | Mean: 0.0000
Negative values: 309 / 122,623

=== SAMPLE DATE: 2022-06-01 (hiking cycle — should have shorts) ===
Non-zero: 461 | Negative: 201 | Positive: 202
Min value: -0.9949 | Max value: 1.0000

Bottom 5 stocks (should be shorts):
WOCKPHARMA.NS   -0.9949
JPPOWER.NS      -0.9942
PNBHOUSING.NS   -0.9899
UJJIVANSFB.NS   -0.9848
HEG.NS          -0.9797
Name: 2022-06-01 00:00:00, dtype: float64


In [40]:
import pandas as pd
import numpy as np
import os

BASE     = "/content/hedge_fund/data/processed"
PORT_DIR = f"{BASE}/portfolio"
os.makedirs(PORT_DIR, exist_ok=True)

SLEEVE_W     = {"momentum": 0.50, "rate": 0.30, "recon": 0.20}
MAX_POS      = 0.05
MIN_POS      = 0.005
MAX_SECTOR   = 0.25
MAX_GROSS    = 1.50
MAX_NET      = 0.40
MIN_NET      = -0.20
VOL_TARGET   = 0.12
VOL_LOOKBACK = 63
LONG_THRESH  = 0.20    # Top signals become longs
SHORT_THRESH = -0.20   # Bottom signals become shorts

print("Loading data...")
mom   = pd.read_csv(f"{BASE}/signals/momentum_signal.csv",  index_col=0, parse_dates=True).fillna(0)
rate  = pd.read_csv(f"{BASE}/signals/rate_signal.csv",      index_col=0, parse_dates=True).fillna(0)
recon = pd.read_csv(f"{BASE}/signals/recon_signal.csv",     index_col=0, parse_dates=True).fillna(0)
rets  = pd.read_csv(f"{BASE}/returns.csv",                  index_col=0, parse_dates=True).fillna(0)
meta  = pd.read_csv(f"{BASE}/universe_metadata.csv")

sector_map = {}
for _, r in meta.iterrows():
    sector_map[r["symbol"]]         = r["sector"]
    sector_map[r["symbol"] + ".NS"] = r["sector"]

trading_days = mom.index[(mom.index >= "2015-01-01") & (mom.index <= "2024-12-31")]
print(f"Trading days: {len(trading_days)}")

all_weights  = []
exposure_log = []
port_vol     = 0.12

for i, date in enumerate(trading_days):
    m  = mom.loc[date]  if date in mom.index  else pd.Series(dtype=float)
    r  = rate.loc[date] if date in rate.index else pd.Series(dtype=float)
    rc = recon.loc[date] if date in recon.index else pd.Series(dtype=float)

    # Align all to common universe
    all_stocks = sorted(set(m.index) | set(r.index) | set(rc.index))
    m  = m.reindex(all_stocks).fillna(0)
    r  = r.reindex(all_stocks).fillna(0)
    rc = rc.reindex(all_stocks).fillna(0)

    # Weighted combination
    combined = (SLEEVE_W["momentum"] * m +
                SLEEVE_W["rate"]     * r +
                SLEEVE_W["recon"]    * rc)

    # ── KEY FIX: use percentile thresholds not absolute threshold ──────────
    # Top 20% of signals = LONG, Bottom 20% = SHORT
    non_zero = combined[combined != 0]
    if len(non_zero) < 10:
        exposure_log.append({"date": date, "gross": 0, "net": 0,
                             "long": 0, "short": 0, "n_positions": 0,
                             "n_longs": 0, "n_shorts": 0, "vol_scale": 1.0})
        continue

    long_thresh  = non_zero.quantile(0.80)   # Top 20%
    short_thresh = non_zero.quantile(0.20)   # Bottom 20%

    longs  = combined[combined >= long_thresh]
    shorts = combined[combined <= short_thresh]

    weights = pd.Series(dtype=float)
    if len(longs) > 0:
        w_long  = longs / longs.sum() * 0.50
        weights = pd.concat([weights, w_long])
    if len(shorts) > 0:
        w_short = shorts / shorts.abs().sum() * (-0.50)
        weights = pd.concat([weights, w_short])

    # Position cap
    weights = weights.clip(-MAX_POS, MAX_POS)

    # Sector cap
    for sector in set(sector_map.get(s, "Other") for s in weights.index):
        sec_stocks = [s for s in weights.index if sector_map.get(s, "Other") == sector]
        sec_exp    = weights[sec_stocks].abs().sum()
        if sec_exp > MAX_SECTOR:
            weights[sec_stocks] *= MAX_SECTOR / sec_exp

    # Vol targeting
    common = [s for s in weights.index if s in rets.columns]
    if len(common) >= 5 and date in rets.index:
        idx = rets.index.get_loc(date)
        if idx > VOL_LOOKBACK:
            ret_win  = rets.iloc[idx - VOL_LOOKBACK:idx][common].fillna(0)
            port_ret = ret_win.values @ weights[common].values
            port_vol = np.std(port_ret) * np.sqrt(252)
            if port_vol > 0.001:
                weights *= np.clip(VOL_TARGET / port_vol, 0.5, 2.0)

    # Gross/net limits
    gross = weights.abs().sum()
    net   = weights.sum()
    if gross > MAX_GROSS:
        weights *= MAX_GROSS / gross
    if net > MAX_NET:
        excess = net - MAX_NET
        n_long = (weights > 0).sum()
        if n_long > 0:
            weights[weights > 0] -= excess / n_long
    if net < MIN_NET:
        excess = MIN_NET - net
        n_short = (weights < 0).sum()
        if n_short > 0:
            weights[weights < 0] += excess / n_short

    # Zero out tiny positions
    weights[weights.abs() < MIN_POS] = 0
    weights = weights[weights != 0]

    gross = weights.abs().sum()
    net   = weights.sum()

    weights.name = date
    all_weights.append(weights)
    exposure_log.append({
        "date":        date,
        "gross":       round(gross, 4),
        "net":         round(net, 4),
        "long":        round(weights[weights > 0].sum(), 4),
        "short":       round(weights[weights < 0].sum(), 4),
        "n_positions": len(weights),
        "n_longs":     int((weights > 0).sum()),
        "n_shorts":    int((weights < 0).sum()),
        "vol_scale":   round(float(np.clip(VOL_TARGET / max(port_vol, 0.001), 0.5, 2.0)), 4),
    })

    if i % 500 == 0:
        print(f"  {date.date()}: {len(weights)} pos | gross={gross:.1%} net={net:.1%} | L={int((weights>0).sum())} S={int((weights<0).sum())}")

print("\nAssembling...")
weights_df  = pd.DataFrame(all_weights).fillna(0)
exposure_df = pd.DataFrame(exposure_log).set_index("date")
weights_df.to_csv(f"{PORT_DIR}/weights_daily.csv")
weights_df.resample("ME").last().to_csv(f"{PORT_DIR}/weights_monthly.csv")
exposure_df.to_csv(f"{PORT_DIR}/exposure_daily.csv")

active = exposure_df[exposure_df["n_positions"] > 0]
print(f"\n✓ Done")
print(f"  Total days:          {len(exposure_df)}")
print(f"  Active trading days: {len(active)}")
print(f"  Avg positions:       {active['n_positions'].mean():.1f}")
print(f"  Avg gross exposure:  {active['gross'].mean():.1%}")
print(f"  Avg net exposure:    {active['net'].mean():.1%}")
print(f"  Avg longs:           {active['n_longs'].mean():.1f}")
print(f"  Avg shorts:          {active['n_shorts'].mean():.1f}")
print(f"\nAnnual breakdown:")
print(active[["gross","net","n_longs","n_shorts"]].resample("YE").mean().round(3))

Loading data...
Trading days: 2608
  2016-12-01: 0 pos | gross=0.0% net=0.0% | L=0 S=0
  2018-11-01: 0 pos | gross=0.0% net=0.0% | L=0 S=0
  2020-10-01: 0 pos | gross=0.0% net=0.0% | L=0 S=0
  2022-09-01: 0 pos | gross=0.0% net=0.0% | L=0 S=0
  2024-08-01: 0 pos | gross=0.0% net=0.0% | L=0 S=0

Assembling...

✓ Done
  Total days:          2608
  Active trading days: 342
  Avg positions:       8.1
  Avg gross exposure:  4.7%
  Avg net exposure:    4.7%
  Avg longs:           8.1
  Avg shorts:          0.0

Annual breakdown:
            gross    net  n_longs  n_shorts
date                                       
2015-12-31  0.015  0.015    2.944       0.0
2016-12-31  0.075  0.075   12.705       0.0
2017-12-31  0.024  0.024    4.633       0.0
2018-12-31    NaN    NaN      NaN       NaN
2019-12-31  0.009  0.009    1.688       0.0
2020-12-31  0.012  0.012    2.357       0.0
2021-12-31    NaN    NaN      NaN       NaN
2022-12-31    NaN    NaN      NaN       NaN
2023-12-31  0.050  0.050    8.5

In [41]:
import pandas as pd
import numpy as np

BASE = "/content/hedge_fund/data/processed"

mom  = pd.read_csv(f"{BASE}/signals/momentum_signal.csv",  index_col=0, parse_dates=True)
rate = pd.read_csv(f"{BASE}/signals/rate_signal.csv",      index_col=0, parse_dates=True)
recon = pd.read_csv(f"{BASE}/signals/recon_signal.csv",    index_col=0, parse_dates=True)

# Check what happens on a specific date
date = pd.Timestamp("2022-06-01")

m  = mom.loc[date].fillna(0)
r  = rate.loc[date].fillna(0)
rc = recon.loc[date].fillna(0)

print(f"Date: {date.date()}")
print(f"Momentum  — non-zero: {(m!=0).sum()} | neg: {(m<0).sum()} | pos: {(m>0).sum()}")
print(f"Rate      — non-zero: {(r!=0).sum()} | neg: {(r<0).sum()} | pos: {(r>0).sum()}")
print(f"Recon     — non-zero: {(rc!=0).sum()} | neg: {(rc<0).sum()} | pos: {(rc>0).sum()}")

# Combined
all_stocks = sorted(set(m.index)|set(r.index)|set(rc.index))
m  = m.reindex(all_stocks).fillna(0)
r  = r.reindex(all_stocks).fillna(0)
rc = rc.reindex(all_stocks).fillna(0)

combined = 0.50*m + 0.30*r + 0.20*rc
print(f"\nCombined — non-zero: {(combined!=0).sum()} | neg: {(combined<0).sum()} | pos: {(combined>0).sum()}")
print(f"Combined min: {combined.min():.4f} | max: {combined.max():.4f}")

non_zero = combined[combined != 0]
long_thresh  = non_zero.quantile(0.80)
short_thresh = non_zero.quantile(0.20)
print(f"\nLong threshold (80th pct):  {long_thresh:.4f}")
print(f"Short threshold (20th pct): {short_thresh:.4f}")
print(f"Stocks above long thresh:   {(combined >= long_thresh).sum()}")
print(f"Stocks below short thresh:  {(combined <= short_thresh).sum()}")

print(f"\nTop 5 longs:")
print(combined.nlargest(5))
print(f"\nBottom 5 shorts:")
print(combined.nsmallest(5))

Date: 2022-06-01
Momentum  — non-zero: 403 | neg: 201 | pos: 202
Rate      — non-zero: 337 | neg: 169 | pos: 168
Recon     — non-zero: 0 | neg: 0 | pos: 0

Combined — non-zero: 472 | neg: 234 | pos: 238
Combined min: -0.7650 | max: 0.7298

Long threshold (80th pct):  0.2741
Short threshold (20th pct): -0.3150
Stocks above long thresh:   95
Stocks below short thresh:  95

Top 5 longs:
USHAMART.NS    0.72985
JWL.NS         0.72480
CGPOWER.NS     0.71720
KPITTECH.NS    0.70960
ZENTEC.NS      0.69950
Name: 2022-06-01 00:00:00, dtype: float64

Bottom 5 shorts:
PNBHOUSING.NS   -0.76495
SAMMAANCAP.NS   -0.75735
RBLBANK.NS      -0.75230
BANKINDIA.NS    -0.74975
IDFCFIRSTB.NS   -0.73205
Name: 2022-06-01 00:00:00, dtype: float64


In [42]:
import pandas as pd
import numpy as np
import os

BASE     = "/content/hedge_fund/data/processed"
PORT_DIR = f"{BASE}/portfolio"
os.makedirs(PORT_DIR, exist_ok=True)

SLEEVE_W     = {"momentum": 0.50, "rate": 0.30, "recon": 0.20}
MAX_POS      = 0.05
MIN_POS      = 0.005
MAX_SECTOR   = 0.25
MAX_GROSS    = 1.50
MAX_NET      = 0.40
MIN_NET      = -0.20
VOL_TARGET   = 0.12
VOL_LOOKBACK = 63

print("Loading data...")
mom   = pd.read_csv(f"{BASE}/signals/momentum_signal.csv",  index_col=0, parse_dates=True).fillna(0)
rate  = pd.read_csv(f"{BASE}/signals/rate_signal.csv",      index_col=0, parse_dates=True).fillna(0)
recon = pd.read_csv(f"{BASE}/signals/recon_signal.csv",     index_col=0, parse_dates=True).fillna(0)
rets  = pd.read_csv(f"{BASE}/returns.csv",                  index_col=0, parse_dates=True).fillna(0)
meta  = pd.read_csv(f"{BASE}/universe_metadata.csv")

sector_map = {}
for _, r in meta.iterrows():
    sector_map[r["symbol"]]         = r["sector"]
    sector_map[r["symbol"] + ".NS"] = r["sector"]

trading_days = mom.index[(mom.index >= "2015-01-01") & (mom.index <= "2024-12-31")]
print(f"Trading days: {len(trading_days)}")

# Pre-build returns index as a dict for fast lookup
rets_index_map = {date: i for i, date in enumerate(rets.index)}

all_weights  = []
exposure_log = []

for i, date in enumerate(trading_days):
    try:
        m  = mom.loc[date].fillna(0)
        r  = rate.loc[date].fillna(0)
        rc = recon.loc[date].fillna(0)

        all_stocks = sorted(set(m.index)|set(r.index)|set(rc.index))
        m  = m.reindex(all_stocks).fillna(0)
        r  = r.reindex(all_stocks).fillna(0)
        rc = rc.reindex(all_stocks).fillna(0)

        combined = 0.50*m + 0.30*r + 0.20*rc
        non_zero = combined[combined != 0]

        if len(non_zero) < 10:
            raise ValueError("Not enough signal")

        long_thresh  = non_zero.quantile(0.80)
        short_thresh = non_zero.quantile(0.20)

        longs  = combined[combined >= long_thresh]
        shorts = combined[combined <= short_thresh]

        weights = pd.Series(dtype=float)
        if len(longs) > 0:
            weights = pd.concat([weights, longs / longs.sum() * 0.50])
        if len(shorts) > 0:
            weights = pd.concat([weights, shorts / shorts.abs().sum() * (-0.50)])

        # Position cap
        weights = weights.clip(-MAX_POS, MAX_POS)

        # Sector cap
        for sector in set(sector_map.get(s, "Other") for s in weights.index):
            sec = [s for s in weights.index if sector_map.get(s,"Other") == sector]
            exp = weights[sec].abs().sum()
            if exp > MAX_SECTOR:
                weights[sec] *= MAX_SECTOR / exp

        # Vol targeting — safe version
        vol_scale = 1.0
        try:
            common = [s for s in weights.index if s in rets.columns]
            if len(common) >= 5 and date in rets_index_map:
                idx     = rets_index_map[date]
                if idx >= VOL_LOOKBACK:
                    ret_win  = rets.iloc[idx-VOL_LOOKBACK:idx][common].values
                    w_arr    = weights[common].values
                    port_ret = ret_win @ w_arr
                    port_vol = np.std(port_ret) * np.sqrt(252)
                    if port_vol > 0.001:
                        vol_scale = float(np.clip(VOL_TARGET / port_vol, 0.5, 2.0))
                        weights  *= vol_scale
        except Exception:
            pass  # Keep unscaled weights if vol calc fails

        # Gross/net limits
        gross = weights.abs().sum()
        net   = weights.sum()
        if gross > MAX_GROSS:
            weights *= MAX_GROSS / gross
            gross    = weights.abs().sum()
            net      = weights.sum()
        if net > MAX_NET:
            n_l = (weights > 0).sum()
            if n_l > 0:
                weights[weights > 0] -= (net - MAX_NET) / n_l
        if net < MIN_NET:
            n_s = (weights < 0).sum()
            if n_s > 0:
                weights[weights < 0] += (MIN_NET - net) / n_s

        # Min position filter
        weights[weights.abs() < MIN_POS] = 0
        weights = weights[weights != 0]

        if len(weights) == 0:
            raise ValueError("No positions after filters")

        gross = weights.abs().sum()
        net   = weights.sum()
        weights.name = date
        all_weights.append(weights)

        exposure_log.append({
            "date": date, "gross": round(gross,4), "net": round(net,4),
            "long": round(weights[weights>0].sum(),4),
            "short": round(weights[weights<0].sum(),4),
            "n_positions": len(weights),
            "n_longs": int((weights>0).sum()),
            "n_shorts": int((weights<0).sum()),
            "vol_scale": round(vol_scale,4),
        })

    except Exception as e:
        exposure_log.append({
            "date": date, "gross": 0, "net": 0, "long": 0, "short": 0,
            "n_positions": 0, "n_longs": 0, "n_shorts": 0, "vol_scale": 1.0,
        })

    if i % 500 == 0:
        n = exposure_log[-1]["n_positions"]
        g = exposure_log[-1]["gross"]
        nl = exposure_log[-1]["n_longs"]
        ns = exposure_log[-1]["n_shorts"]
        print(f"  {date.date()}: {n} pos | gross={g:.1%} | L={nl} S={ns}")

print("\nAssembling...")
weights_df  = pd.DataFrame(all_weights).fillna(0)
exposure_df = pd.DataFrame(exposure_log).set_index("date")

weights_df.to_csv(f"{PORT_DIR}/weights_daily.csv")
weights_df.resample("ME").last().to_csv(f"{PORT_DIR}/weights_monthly.csv")
exposure_df.to_csv(f"{PORT_DIR}/exposure_daily.csv")

active = exposure_df[exposure_df["n_positions"] > 0]
print(f"\n✓ Portfolio construction complete")
print(f"  Total days:          {len(exposure_df)}")
print(f"  Active trading days: {len(active)}")
print(f"  Avg positions:       {active['n_positions'].mean():.1f}")
print(f"  Avg gross exposure:  {active['gross'].mean():.1%}")
print(f"  Avg net exposure:    {active['net'].mean():.1%}")
print(f"  Avg longs:           {active['n_longs'].mean():.1f}")
print(f"  Avg shorts:          {active['n_shorts'].mean():.1f}")
print(f"\nAnnual breakdown:")
print(active[["gross","net","n_longs","n_shorts"]].resample("YE").mean().round(3))

Loading data...
Trading days: 2608
  2015-01-01: 0 pos | gross=0.0% | L=0 S=0
  2016-12-01: 0 pos | gross=0.0% | L=0 S=0
  2018-11-01: 0 pos | gross=0.0% | L=0 S=0
  2020-10-01: 0 pos | gross=0.0% | L=0 S=0
  2022-09-01: 0 pos | gross=0.0% | L=0 S=0
  2024-08-01: 0 pos | gross=0.0% | L=0 S=0

Assembling...

✓ Portfolio construction complete
  Total days:          2608
  Active trading days: 342
  Avg positions:       8.3
  Avg gross exposure:  4.8%
  Avg net exposure:    4.8%
  Avg longs:           8.3
  Avg shorts:          0.0

Annual breakdown:
            gross    net  n_longs  n_shorts
date                                       
2015-12-31  0.015  0.015    2.944       0.0
2016-12-31  0.075  0.075   12.705       0.0
2017-12-31  0.024  0.024    4.633       0.0
2018-12-31    NaN    NaN      NaN       NaN
2019-12-31  0.009  0.009    1.688       0.0
2020-12-31  0.012  0.012    2.357       0.0
2021-12-31    NaN    NaN      NaN       NaN
2022-12-31    NaN    NaN      NaN       NaN
2023-1

In [43]:
import pandas as pd
import numpy as np

BASE = "/content/hedge_fund/data/processed"

mom   = pd.read_csv(f"{BASE}/signals/momentum_signal.csv",  index_col=0, parse_dates=True).fillna(0)
rate  = pd.read_csv(f"{BASE}/signals/rate_signal.csv",      index_col=0, parse_dates=True).fillna(0)
recon = pd.read_csv(f"{BASE}/signals/recon_signal.csv",     index_col=0, parse_dates=True).fillna(0)
rets  = pd.read_csv(f"{BASE}/returns.csv",                  index_col=0, parse_dates=True).fillna(0)
meta  = pd.read_csv(f"{BASE}/universe_metadata.csv")

sector_map = {}
for _, r in meta.iterrows():
    sector_map[r["symbol"]]         = r["sector"]
    sector_map[r["symbol"] + ".NS"] = r["sector"]

# Test exactly ONE date manually — step by step
date = pd.Timestamp("2022-06-01")
print(f"Testing date: {date}")
print(f"Date in mom index: {date in mom.index}")
print(f"Date in rets index: {date in rets.index}")

m  = mom.loc[date].fillna(0)
r  = rate.loc[date].fillna(0)
rc = recon.loc[date].fillna(0)
print(f"Signals loaded: m={len(m)} r={len(r)} rc={len(rc)}")

all_stocks = sorted(set(m.index)|set(r.index)|set(rc.index))
m  = m.reindex(all_stocks).fillna(0)
r  = r.reindex(all_stocks).fillna(0)
rc = rc.reindex(all_stocks).fillna(0)

combined = 0.50*m + 0.30*r + 0.20*rc
non_zero = combined[combined != 0]
print(f"Combined non-zero: {len(non_zero)}")

long_thresh  = non_zero.quantile(0.80)
short_thresh = non_zero.quantile(0.20)
longs  = combined[combined >= long_thresh]
shorts = combined[combined <= short_thresh]
print(f"Longs: {len(longs)} | Shorts: {len(shorts)}")

weights = pd.Series(dtype=float)
weights = pd.concat([weights, longs / longs.sum() * 0.50])
weights = pd.concat([weights, shorts / shorts.abs().sum() * (-0.50)])
print(f"Weights before cap: {len(weights)} | gross={weights.abs().sum():.2%}")
print(f"Longs: {(weights>0).sum()} | Shorts: {(weights<0).sum()}")

weights = weights.clip(-0.05, 0.05)
print(f"After position cap: gross={weights.abs().sum():.2%}")

# Vol targeting
rets_index_map = {d: i for i, d in enumerate(rets.index)}
common = [s for s in weights.index if s in rets.columns]
print(f"Common stocks for vol: {len(common)}")
idx = rets_index_map.get(date, -1)
print(f"Returns index for date: {idx}")

if idx >= 63:
    ret_win  = rets.iloc[idx-63:idx][common].values
    w_arr    = weights[common].values
    port_ret = ret_win @ w_arr
    port_vol = np.std(port_ret) * np.sqrt(252)
    print(f"Portfolio vol: {port_vol:.2%}")
    vol_scale = float(np.clip(0.12 / port_vol, 0.5, 2.0))
    print(f"Vol scale: {vol_scale:.2f}x")
    weights *= vol_scale

weights[weights.abs() < 0.005] = 0
weights = weights[weights != 0]
print(f"\nFinal weights: {len(weights)} positions")
print(f"Gross: {weights.abs().sum():.2%} | Net: {weights.sum():.2%}")
print(f"Longs: {(weights>0).sum()} | Shorts: {(weights<0).sum()}")

Testing date: 2022-06-01 00:00:00
Date in mom index: True
Date in rets index: True
Signals loaded: m=461 r=341 rc=47
Combined non-zero: 472
Longs: 95 | Shorts: 95
Weights before cap: 190 | gross=100.00%
Longs: 190 | Shorts: 0
After position cap: gross=100.00%
Common stocks for vol: 190
Returns index for date: 2195
Portfolio vol: 21.14%
Vol scale: 0.57x

Final weights: 0 positions
Gross: 0.00% | Net: 0.00%
Longs: 0 | Shorts: 0


In [44]:
import pandas as pd
import numpy as np
import os

BASE     = "/content/hedge_fund/data/processed"
PORT_DIR = f"{BASE}/portfolio"
os.makedirs(PORT_DIR, exist_ok=True)

SLEEVE_W     = {"momentum": 0.50, "rate": 0.30, "recon": 0.20}
MAX_POS      = 0.05
MIN_POS      = 0.003    # Lowered from 0.005
MAX_SECTOR   = 0.25
MAX_GROSS    = 1.50
MAX_NET      = 0.40
MIN_NET      = -0.20
VOL_TARGET   = 0.12
VOL_LOOKBACK = 63
TOP_N        = 20       # Take top 20 longs and top 20 shorts only

print("Loading data...")
mom   = pd.read_csv(f"{BASE}/signals/momentum_signal.csv",  index_col=0, parse_dates=True).fillna(0)
rate  = pd.read_csv(f"{BASE}/signals/rate_signal.csv",      index_col=0, parse_dates=True).fillna(0)
recon = pd.read_csv(f"{BASE}/signals/recon_signal.csv",     index_col=0, parse_dates=True).fillna(0)
rets  = pd.read_csv(f"{BASE}/returns.csv",                  index_col=0, parse_dates=True).fillna(0)
meta  = pd.read_csv(f"{BASE}/universe_metadata.csv")

sector_map = {}
for _, r in meta.iterrows():
    sector_map[r["symbol"]]         = r["sector"]
    sector_map[r["symbol"] + ".NS"] = r["sector"]

rets_index_map = {d: i for i, d in enumerate(rets.index)}
trading_days   = mom.index[(mom.index >= "2015-01-01") & (mom.index <= "2024-12-31")]
print(f"Trading days: {len(trading_days)}")

all_weights  = []
exposure_log = []

for i, date in enumerate(trading_days):
    try:
        m  = mom.loc[date].fillna(0)
        r  = rate.loc[date].fillna(0)
        rc = recon.loc[date].fillna(0)

        all_stocks = sorted(set(m.index)|set(r.index)|set(rc.index))
        m  = m.reindex(all_stocks).fillna(0)
        r  = r.reindex(all_stocks).fillna(0)
        rc = rc.reindex(all_stocks).fillna(0)

        combined = 0.50*m + 0.30*r + 0.20*rc
        non_zero = combined[combined != 0]
        if len(non_zero) < 10:
            raise ValueError("insufficient signal")

        # Take top N longs and top N shorts
        longs  = combined.nlargest(TOP_N)
        shorts = combined.nsmallest(TOP_N)

        # Only keep genuine longs (positive) and genuine shorts (negative)
        longs  = longs[longs > 0]
        shorts = shorts[shorts < 0]

        if len(longs) == 0 and len(shorts) == 0:
            raise ValueError("no valid positions")

        weights = pd.Series(dtype=float)

        if len(longs) > 0:
            # Equal weight longs, target 50% gross long
            w_long = pd.Series(0.50 / len(longs), index=longs.index)
            weights = pd.concat([weights, w_long])

        if len(shorts) > 0:
            # Equal weight shorts, target -50% gross short — NEGATIVE weights
            w_short = pd.Series(-0.50 / len(shorts), index=shorts.index)
            weights = pd.concat([weights, w_short])

        # Verify signs are correct
        assert (weights[weights > 0] > 0).all(), "Long weights should be positive"
        assert (weights[weights < 0] < 0).all(), "Short weights should be negative"

        # Position cap
        weights = weights.clip(-MAX_POS, MAX_POS)

        # Sector cap
        for sector in set(sector_map.get(s, "Other") for s in weights.index):
            sec = [s for s in weights.index if sector_map.get(s,"Other") == sector]
            exp = weights[sec].abs().sum()
            if exp > MAX_SECTOR:
                weights[sec] *= MAX_SECTOR / exp

        # Vol targeting
        vol_scale = 1.0
        common = [s for s in weights.index if s in rets.columns]
        if len(common) >= 5:
            idx = rets_index_map.get(date, -1)
            if idx >= VOL_LOOKBACK:
                ret_win  = rets.iloc[idx-VOL_LOOKBACK:idx][common].values
                w_arr    = weights[common].values
                port_ret = ret_win @ w_arr
                port_vol = np.std(port_ret) * np.sqrt(252)
                if port_vol > 0.001:
                    vol_scale = float(np.clip(VOL_TARGET / port_vol, 0.5, 2.0))
                    weights  *= vol_scale

        # Gross/net limits
        gross = weights.abs().sum()
        net   = weights.sum()
        if gross > MAX_GROSS:
            weights *= MAX_GROSS / gross
        net = weights.sum()
        if net > MAX_NET:
            n_l = (weights > 0).sum()
            if n_l > 0: weights[weights > 0] -= (net - MAX_NET) / n_l
        if net < MIN_NET:
            n_s = (weights < 0).sum()
            if n_s > 0: weights[weights < 0] += (MIN_NET - net) / n_s

        # Min position — after vol scaling positions are smaller, use lower threshold
        weights[weights.abs() < MIN_POS] = 0
        weights = weights[weights != 0]

        if len(weights) == 0:
            raise ValueError("all positions below minimum")

        gross = weights.abs().sum()
        net   = weights.sum()
        weights.name = date
        all_weights.append(weights)

        exposure_log.append({
            "date": date, "gross": round(gross,4), "net": round(net,4),
            "long": round(weights[weights>0].sum(),4),
            "short": round(weights[weights<0].sum(),4),
            "n_positions": len(weights),
            "n_longs": int((weights>0).sum()),
            "n_shorts": int((weights<0).sum()),
            "vol_scale": round(vol_scale,4),
        })

    except Exception as e:
        exposure_log.append({
            "date": date, "gross": 0, "net": 0, "long": 0, "short": 0,
            "n_positions": 0, "n_longs": 0, "n_shorts": 0, "vol_scale": 1.0,
        })

    if i % 500 == 0:
        n  = exposure_log[-1]["n_positions"]
        g  = exposure_log[-1]["gross"]
        nl = exposure_log[-1]["n_longs"]
        ns = exposure_log[-1]["n_shorts"]
        print(f"  {date.date()}: {n} pos | gross={g:.1%} | L={nl} S={ns}")

print("\nAssembling...")
weights_df  = pd.DataFrame(all_weights).fillna(0)
exposure_df = pd.DataFrame(exposure_log).set_index("date")

weights_df.to_csv(f"{PORT_DIR}/weights_daily.csv")
weights_df.resample("ME").last().to_csv(f"{PORT_DIR}/weights_monthly.csv")
exposure_df.to_csv(f"{PORT_DIR}/exposure_daily.csv")

active = exposure_df[exposure_df["n_positions"] > 0]
print(f"\n✓ Portfolio construction complete")
print(f"  Total days:          {len(exposure_df)}")
print(f"  Active trading days: {len(active)}")
print(f"  Avg positions:       {active['n_positions'].mean():.1f}")
print(f"  Avg gross exposure:  {active['gross'].mean():.1%}")
print(f"  Avg net exposure:    {active['net'].mean():.1%}")
print(f"  Avg longs:           {active['n_longs'].mean():.1f}")
print(f"  Avg shorts:          {active['n_shorts'].mean():.1f}")
print(f"\nAnnual breakdown:")
print(active[["gross","net","n_longs","n_shorts"]].resample("YE").mean().round(3))

Loading data...
Trading days: 2608
  2015-01-01: 0 pos | gross=0.0% | L=0 S=0
  2016-12-01: 40 pos | gross=104.0% | L=20 S=20
  2018-11-01: 40 pos | gross=114.7% | L=20 S=20
  2020-10-01: 40 pos | gross=148.6% | L=20 S=20
  2022-09-01: 40 pos | gross=136.8% | L=20 S=20
  2024-08-01: 40 pos | gross=101.2% | L=20 S=20

Assembling...

✓ Portfolio construction complete
  Total days:          2608
  Active trading days: 2545
  Avg positions:       40.0
  Avg gross exposure:  129.0%
  Avg net exposure:    -0.4%
  Avg longs:           20.0
  Avg shorts:          20.0

Annual breakdown:
            gross    net  n_longs  n_shorts
date                                       
2015-12-31  1.316 -0.001     20.0      20.0
2016-12-31  1.388  0.136     20.0      20.0
2017-12-31  1.339  0.030     20.0      20.0
2018-12-31  1.242  0.002     20.0      20.0
2019-12-31  1.199 -0.029     20.0      20.0
2020-12-31  1.191  0.081     20.0      20.0
2021-12-31  1.333  0.007     20.0      20.0
2022-12-31  1.387 

In [45]:
import shutil
from google.colab import files

shutil.make_archive("/content/hedge_fund_portfolio", "zip",
                    "/content/hedge_fund/data/processed/portfolio")
files.download("/content/hedge_fund_portfolio.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [46]:
import pandas as pd

weights  = pd.read_csv("/content/hedge_fund/data/processed/portfolio/weights_daily.csv",
                       index_col=0, parse_dates=True)
exposure = pd.read_csv("/content/hedge_fund/data/processed/portfolio/exposure_daily.csv",
                       index_col=0, parse_dates=True)

# Most frequently held longs and shorts
long_freq  = (weights > 0).mean().nlargest(5)
short_freq = (weights < 0).mean().nlargest(5)

print("Most frequently LONG:")
for s, f in long_freq.items():
    print(f"  {s.replace('.NS',''):<20} {f:.1%} of days")

print("\nMost frequently SHORT:")
for s, f in short_freq.items():
    print(f"  {s.replace('.NS',''):<20} {f:.1%} of days")

print(f"\nLatest portfolio (2024-12-30):")
latest = weights.loc["2024-12-30"]
latest = latest[latest != 0].sort_values(ascending=False)
print("Top 5 longs:")
print(latest.head(5).apply(lambda x: f"{x:.2%}"))
print("Top 5 shorts:")
print(latest.tail(5).apply(lambda x: f"{x:.2%}"))

Most frequently LONG:
  BAJFINANCE           26.0% of days
  KEI                  25.5% of days
  CGCL                 25.5% of days
  TITAGARH             24.8% of days
  MUTHOOTFIN           24.6% of days

Most frequently SHORT:
  JPPOWER              64.8% of days
  GPIL                 34.0% of days
  INOXWIND             32.6% of days
  WOCKPHARMA           26.1% of days
  GVT&D                25.7% of days

Latest portfolio (2024-12-30):
Top 5 longs:
PGEL.NS        3.85%
ANANTRAJ.NS    3.85%
NAVA.NS        3.85%
HSCL.NS        3.85%
TARIL.NS       3.85%
Name: 2024-12-30 00:00:00, dtype: object
Top 5 shorts:
RHIM.NS          -3.85%
UJJIVANSFB.NS    -3.85%
ZEEL.NS          -3.85%
SYRMA.NS         -3.85%
CERA.NS          -3.85%
Name: 2024-12-30 00:00:00, dtype: object


In [47]:
import shutil
from google.colab import files

shutil.make_archive("/content/hedge_fund_portfolio", "zip",
                    "/content/hedge_fund/data/processed/portfolio")
files.download("/content/hedge_fund_portfolio.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [48]:
import os, shutil
os.makedirs("/content/hedge_fund/scripts/backtest", exist_ok=True)

if os.path.exists("/content/backtest_engine.py"):
    shutil.copy("/content/backtest_engine.py",
                "/content/hedge_fund/scripts/backtest/backtest_engine.py")
    print("✓ Moved backtest_engine.py")
else:
    print("✗ Upload backtest_engine.py first")

✓ Moved backtest_engine.py


In [50]:
import pandas as pd
import numpy as np
import os, json

BASE         = "/content/hedge_fund/data/processed"
PORT_DIR     = f"{BASE}/portfolio"
BACKTEST_DIR = f"{BASE}/backtest"
os.makedirs(BACKTEST_DIR, exist_ok=True)

STARTING_NAV       = 100.0
RF_DAILY           = 0.065 / 252
SHORT_FINANCE_RATE = 0.065 / 252
BROKERAGE_RT       = 0.0007   # round-trip brokerage + exchange
STT_PCT            = 0.0005   # blended STT estimate

print("Loading data...")
weights   = pd.read_csv(f"{PORT_DIR}/weights_daily.csv",       index_col=0, parse_dates=True)
returns   = pd.read_csv(f"{BASE}/returns.csv",                  index_col=0, parse_dates=True)
benchmark = pd.read_csv(f"{BASE}/benchmark_returns.csv",        index_col=0, parse_dates=True)

print(f"Weights:   {weights.shape}")
print(f"Returns:   {returns.shape}")
print(f"Benchmark: {benchmark.shape}")

# ── Run simulation ─────────────────────────────────────────────────────────────
common_dates = weights.index.intersection(returns.index)
common_dates = common_dates[(common_dates >= "2015-01-01") & (common_dates <= "2024-12-31")]
print(f"Backtest days: {len(common_dates)}")

nav          = STARTING_NAV
nav_log      = []
ret_log      = []
prev_weights = None

for i, date in enumerate(common_dates):
    w = weights.loc[date].fillna(0)
    w = w[w != 0]

    if i == 0:
        prev_weights = w
        nav_log.append({"date": date, "nav": nav})
        ret_log.append({"date": date, "return": 0.0, "gross_pnl": 0.0,
                        "long_pnl": 0.0, "short_pnl": 0.0,
                        "tcost": 0.0, "fin_cost": 0.0})
        continue

    # Apply YESTERDAY's weights to TODAY's returns (no look-ahead)
    w_prev        = prev_weights
    common_stocks = [s for s in w_prev.index if s in returns.columns]

    gross_pnl = 0.0
    long_pnl  = 0.0
    short_pnl = 0.0

    if len(common_stocks) > 0:
        r_today       = returns.loc[date, common_stocks].fillna(0)
        w_aligned     = w_prev.reindex(common_stocks).fillna(0)
        gross_pnl     = float((w_aligned * r_today).sum())
        long_pnl      = float((w_aligned[w_aligned > 0] * r_today[w_aligned[w_aligned > 0].index]).sum())
        short_pnl     = float((w_aligned[w_aligned < 0] * r_today[w_aligned[w_aligned < 0].index]).sum())

    # Turnover and transaction costs
    all_s    = w.index.union(w_prev.index)
    w_t      = w.reindex(all_s).fillna(0)
    w_p      = w_prev.reindex(all_s).fillna(0)
    turnover = float((w_t - w_p).abs().sum() / 2)
    tcost    = (BROKERAGE_RT + STT_PCT) * turnover

    # Short financing
    fin_cost = float(w_prev[w_prev < 0].abs().sum()) * SHORT_FINANCE_RATE

    net_ret = gross_pnl - tcost - fin_cost
    nav     = nav * (1 + net_ret)

    nav_log.append({"date": date, "nav": nav})
    ret_log.append({"date": date, "return": net_ret, "gross_pnl": gross_pnl,
                    "long_pnl": long_pnl, "short_pnl": short_pnl,
                    "tcost": tcost, "fin_cost": fin_cost})

    prev_weights = w

    if i % 500 == 0:
        print(f"  {date.date()}: NAV={nav:.2f} | ret={net_ret:.2%} | turnover={turnover:.1%}")

# ── Assemble results ───────────────────────────────────────────────────────────
nav_df = pd.DataFrame(nav_log).set_index("date")
ret_df = pd.DataFrame(ret_log).set_index("date")

# Benchmark NAV
bench_nav = pd.DataFrame(index=benchmark.index)
for col in benchmark.columns:
    n = STARTING_NAV
    ns = []
    for r in benchmark[col].fillna(0):
        n = n * (1 + r)
        ns.append(n)
    bench_nav[col] = ns

bench_nav = bench_nav.loc[common_dates[0]:common_dates[-1]]
combined_nav = nav_df.copy()
combined_nav.columns = ["Strategy"]
for col in bench_nav.columns:
    combined_nav[col] = bench_nav[col]

combined_nav.to_csv(f"{BACKTEST_DIR}/nav.csv")
ret_df.to_csv(f"{BACKTEST_DIR}/returns.csv")

# ── Statistics ─────────────────────────────────────────────────────────────────
ret   = ret_df["return"]
total = (nav_df["nav"].iloc[-1] / STARTING_NAV) - 1
years = len(ret) / 252
cagr  = (1 + total) ** (1 / years) - 1
vol   = ret.std() * np.sqrt(252)
sharpe = ((ret - RF_DAILY).mean() / ret.std()) * np.sqrt(252)

dd     = (nav_df["nav"] / nav_df["nav"].cummax()) - 1
max_dd = dd.min()

sortino_down = ret[ret < RF_DAILY].std()
sortino = ((ret - RF_DAILY).mean() / sortino_down) * np.sqrt(252) if sortino_down > 0 else 0

monthly = (1 + ret).resample("ME").prod() - 1
annual  = (1 + ret).resample("YE").prod() - 1
calmar  = cagr / abs(max_dd) if max_dd != 0 else 0
win_rate = (ret > 0).mean()

print(f"\n{'='*55}")
print(f"  BACKTEST RESULTS — 2015 to 2024")
print(f"{'='*55}")
print(f"  Starting NAV:        ₹{STARTING_NAV:.2f}")
print(f"  Ending NAV:          ₹{nav_df['nav'].iloc[-1]:.2f}")
print(f"  Total Return:        {total:.1%}")
print(f"  CAGR:                {cagr:.1%}")
print(f"  Ann. Volatility:     {vol:.1%}")
print(f"  Sharpe Ratio:        {sharpe:.2f}")
print(f"  Sortino Ratio:       {sortino:.2f}")
print(f"  Calmar Ratio:        {calmar:.2f}")
print(f"  Max Drawdown:        {max_dd:.1%}")
print(f"  Win Rate:            {win_rate:.1%}")
print(f"  Best Month:          {monthly.max():.1%}")
print(f"  Worst Month:         {monthly.min():.1%}")
print(f"  % Positive Months:   {(monthly > 0).mean():.1%}")
print(f"\n  Annual Returns:")
for yr, r in annual.items():
    bar  = "█" * min(int(abs(r) * 100), 40)
    sign = "+" if r >= 0 else ""
    print(f"    {yr.year}: {sign}{r:.1%}  {bar}")

print(f"\n  vs Benchmarks (CAGR):")
for col in bench_nav.columns:
    b_ret   = (bench_nav[col].iloc[-1] / STARTING_NAV) - 1
    b_cagr  = (1 + b_ret) ** (1 / years) - 1
    print(f"    {col}: {b_cagr:.1%}")

print(f"{'='*55}")

# Save stats
stats = {
    "total_return": round(float(total), 4),
    "cagr": round(float(cagr), 4),
    "ann_volatility": round(float(vol), 4),
    "sharpe_ratio": round(float(sharpe), 3),
    "sortino_ratio": round(float(sortino), 3),
    "calmar_ratio": round(float(calmar), 3),
    "max_drawdown": round(float(max_dd), 4),
    "win_rate": round(float(win_rate), 4),
    "ending_nav": round(float(nav_df["nav"].iloc[-1]), 2),
    "annual_returns": {str(k.year): round(float(v), 4) for k, v in annual.items()},
}
with open(f"{BACKTEST_DIR}/backtest_stats.json", "w") as f:
    json.dump(stats, f, indent=2)

print(f"\n✓ Backtest complete — files saved to {BACKTEST_DIR}")

Loading data...
Weights:   (2545, 396)
Returns:   (2869, 461)
Benchmark: (2704, 2)
Backtest days: 2545
  2017-02-28: NAV=111.22 | ret=-0.05% | turnover=33.7%
  2019-01-29: NAV=153.51 | ret=-0.03% | turnover=1.5%
  2020-12-29: NAV=167.18 | ret=-0.97% | turnover=0.2%
  2022-11-29: NAV=207.54 | ret=-0.98% | turnover=0.0%
  2024-10-29: NAV=229.49 | ret=0.17% | turnover=0.0%

  BACKTEST RESULTS — 2015 to 2024
  Starting NAV:        ₹100.00
  Ending NAV:          ₹235.21
  Total Return:        135.2%
  CAGR:                8.8%
  Ann. Volatility:     11.3%
  Sharpe Ratio:        0.23
  Sortino Ratio:       0.33
  Calmar Ratio:        0.48
  Max Drawdown:        -18.2%
  Win Rate:            49.9%
  Best Month:          10.7%
  Worst Month:         -8.2%
  % Positive Months:   57.6%

  Annual Returns:
    2015: +12.4%  ████████████
    2016: -5.7%  █████
    2017: +32.4%  ████████████████████████████████
    2018: +5.1%  █████
    2019: +11.3%  ███████████
    2020: +3.7%  ███
    2021: +25.6

In [51]:
import shutil
from google.colab import files

shutil.make_archive("/content/hedge_fund_backtest", "zip",
                    "/content/hedge_fund/data/processed/backtest")
files.download("/content/hedge_fund_backtest.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [52]:
import pandas as pd

nav = pd.read_csv("/content/hedge_fund/data/processed/backtest/nav.csv",
                  index_col=0, parse_dates=True)

print(f"{'Year':<6} {'Strategy':>10} {'NIFTY500':>10} {'Difference':>12}")
print("-" * 42)

annual = nav.resample("YE").last()
for yr, row in annual.iterrows():
    prev = nav.loc[:yr].iloc[-252] if len(nav.loc[:yr]) > 252 else nav.iloc[0]
    s_ret  = (row["Strategy"] / prev["Strategy"]) - 1
    b_ret  = (row["NIFTY500"] / prev["NIFTY500"]) - 1
    diff   = s_ret - b_ret
    flag   = "✓" if s_ret > b_ret else "✗"
    print(f"{yr.year:<6} {s_ret:>9.1%} {b_ret:>10.1%} {diff:>11.1%}  {flag}")

Year     Strategy   NIFTY500   Difference
------------------------------------------
2015       12.4%      -3.6%       16.0%  ✓
2016       -5.4%       9.5%      -14.9%  ✗
2017       29.4%      31.2%       -1.8%  ✗
2018        0.1%      -5.1%        5.2%  ✓
2019       12.1%       9.1%        3.1%  ✓
2020        8.4%      14.2%       -5.7%  ✗
2021       26.7%      24.2%        2.4%  ✓
2022       -4.4%      -1.7%       -2.6%  ✗
2023       -0.8%      27.2%      -28.0%  ✗
2024       15.8%      13.2%        2.6%  ✓


In [53]:
import pandas as pd
import numpy as np
from scipy import stats
import os, json

BASE         = "/content/hedge_fund/data/processed"
BACKTEST_DIR = f"{BASE}/backtest"
SIGNAL_DIR   = f"{BASE}/signals"
PORT_DIR     = f"{BASE}/portfolio"
RISK_DIR     = f"{BASE}/risk"
os.makedirs(RISK_DIR, exist_ok=True)

RF_DAILY = 0.065 / 252

print("Loading data...")
nav     = pd.read_csv(f"{BACKTEST_DIR}/nav.csv",      index_col=0, parse_dates=True)
bt_ret  = pd.read_csv(f"{BACKTEST_DIR}/returns.csv",  index_col=0, parse_dates=True)
weights = pd.read_csv(f"{PORT_DIR}/weights_daily.csv",index_col=0, parse_dates=True)
returns = pd.read_csv(f"{BASE}/returns.csv",           index_col=0, parse_dates=True)
mom_sig = pd.read_csv(f"{SIGNAL_DIR}/momentum_signal.csv", index_col=0, parse_dates=True).fillna(0)
rate_sig= pd.read_csv(f"{SIGNAL_DIR}/rate_signal.csv",     index_col=0, parse_dates=True).fillna(0)
recon_sig=pd.read_csv(f"{SIGNAL_DIR}/recon_signal.csv",    index_col=0, parse_dates=True).fillna(0)
print("Loaded.")

# ── 1. Sleeve Attribution ──────────────────────────────────────────────────────
print("\nComputing sleeve attribution...")
SLEEVE_W = {"momentum": 0.50, "rate": 0.30, "recon": 0.20}
common_dates = weights.index.intersection(returns.index)
common_dates = common_dates[(common_dates >= "2015-01-01") & (common_dates <= "2024-12-31")]

attr_records = []
for i, date in enumerate(common_dates[1:], 1):
    prev = common_dates[i-1]
    w_prev = weights.loc[prev].fillna(0)
    w_prev = w_prev[w_prev != 0]
    if len(w_prev) == 0: continue
    cs = [s for s in w_prev.index if s in returns.columns]
    if len(cs) == 0: continue
    r_today   = returns.loc[date, cs].fillna(0)
    w_aligned = w_prev.reindex(cs).fillna(0)
    def gs(df, d, stocks):
        if d not in df.index: return pd.Series(0, index=stocks)
        return df.loc[d].reindex(stocks).fillna(0)
    m_s  = gs(mom_sig,   prev, cs)
    r_s  = gs(rate_sig,  prev, cs)
    rc_s = gs(recon_sig, prev, cs)
    total = (SLEEVE_W["momentum"]*m_s.abs() + SLEEVE_W["rate"]*r_s.abs() + SLEEVE_W["recon"]*rc_s.abs()).replace(0, np.nan)
    stock_pnl = w_aligned * r_today
    attr_records.append({
        "date":      date,
        "total_pnl": float(stock_pnl.sum()),
        "mom_pnl":   float((stock_pnl * (SLEEVE_W["momentum"]*m_s.abs()/total).fillna(0)).sum()),
        "rate_pnl":  float((stock_pnl * (SLEEVE_W["rate"]*r_s.abs()/total).fillna(0)).sum()),
        "recon_pnl": float((stock_pnl * (SLEEVE_W["recon"]*rc_s.abs()/total).fillna(0)).sum()),
        "long_pnl":  float(stock_pnl[w_aligned > 0].sum()),
        "short_pnl": float(stock_pnl[w_aligned < 0].sum()),
    })

attr_df = pd.DataFrame(attr_records).set_index("date")
attr_df.to_csv(f"{RISK_DIR}/sleeve_attribution.csv")
attr_df.cumsum().to_csv(f"{RISK_DIR}/sleeve_attribution_cumulative.csv")
print(f"  Momentum P&L:  {attr_df['mom_pnl'].sum():.2%}")
print(f"  Rate P&L:      {attr_df['rate_pnl'].sum():.2%}")
print(f"  Recon P&L:     {attr_df['recon_pnl'].sum():.2%}")
print(f"  Long book:     {attr_df['long_pnl'].sum():.2%}")
print(f"  Short book:    {attr_df['short_pnl'].sum():.2%}")

# ── 2. Factor Exposure ─────────────────────────────────────────────────────────
print("\nComputing factor exposure...")
ret = bt_ret["return"].dropna()
mkt = nav["NIFTY500"].pct_change().dropna()
common = ret.index.intersection(mkt.index)
y = (ret.loc[common] - RF_DAILY).values
x = (mkt.loc[common] - RF_DAILY).values
slope, intercept, r_val, p_val, std_err = stats.linregress(x, y)
alpha_annual = intercept * 252
residuals    = y - (intercept + slope * x)
ir = (residuals.mean() / residuals.std()) * np.sqrt(252)
print(f"  Market beta:      {slope:.3f}")
print(f"  Jensen's alpha:   {alpha_annual:.2%} p.a.")
print(f"  R-squared:        {r_val**2:.3f}")
print(f"  Info ratio:       {ir:.3f}")

# ── 3. Drawdown Table ──────────────────────────────────────────────────────────
print("\nComputing drawdowns...")
strat_nav = nav["Strategy"]
cummax    = strat_nav.cummax()
dd_series = (strat_nav - cummax) / cummax
dd_series.to_csv(f"{RISK_DIR}/drawdown_series.csv")

drawdowns = []
in_dd, start, trough_val, trough_dt = False, None, 0, None
for date, val in dd_series.items():
    if val < -0.01 and not in_dd:
        in_dd = True; start = date; trough_val = val; trough_dt = date
    elif in_dd:
        if val < trough_val: trough_val = val; trough_dt = date
        if val >= -0.001:
            drawdowns.append({"start": start, "trough": trough_dt, "recovery": date,
                              "depth": round(trough_val,4),
                              "duration_days": (trough_dt-start).days,
                              "recovery_days": (date-trough_dt).days})
            in_dd = False
dd_df = pd.DataFrame(drawdowns).sort_values("depth").reset_index(drop=True)
dd_df.to_csv(f"{RISK_DIR}/drawdown_table.csv", index=False)
print(f"  Drawdown periods: {len(dd_df)}")
print(f"\n  Top 5 worst drawdowns:")
print(dd_df[["start","trough","recovery","depth","duration_days","recovery_days"]].head(5).to_string(index=False))

# ── 4. Rolling Metrics ─────────────────────────────────────────────────────────
print("\nComputing rolling metrics...")
W = 252
roll = pd.DataFrame(index=ret.index)
roll["sharpe"] = ((ret - RF_DAILY).rolling(W).mean() / ret.rolling(W).std()) * np.sqrt(252)
roll["vol"]    = ret.rolling(W).std() * np.sqrt(252)
mkt_aligned    = mkt.reindex(ret.index).fillna(0)
cov            = ret.rolling(W).cov(mkt_aligned)
var_m          = mkt_aligned.rolling(W).var().replace(0, np.nan)
roll["beta"]   = cov / var_m
roll.to_csv(f"{RISK_DIR}/rolling_metrics.csv")
print(f"  Avg rolling Sharpe: {roll['sharpe'].mean():.3f}")
print(f"  Avg rolling vol:    {roll['vol'].mean():.2%}")
print(f"  Avg rolling beta:   {roll['beta'].mean():.3f}")
print(f"  % time Sharpe > 1:  {(roll['sharpe'] > 1).mean():.1%}")

# ── 5. Return Distribution ─────────────────────────────────────────────────────
print("\nReturn distribution:")
var95  = float(np.percentile(ret, 5))
cvar95 = float(ret[ret <= var95].mean())
print(f"  Skewness:     {ret.skew():.3f}")
print(f"  Kurtosis:     {ret.kurtosis():.3f}")
print(f"  VaR 95%:      {var95:.2%} (daily)")
print(f"  CVaR 95%:     {cvar95:.2%} (daily)")

# ── Save risk report ───────────────────────────────────────────────────────────
risk_report = {
    "factor_exposure": {
        "market_beta": round(float(slope),4),
        "jensens_alpha": round(float(alpha_annual),4),
        "r_squared": round(float(r_val**2),4),
        "information_ratio": round(float(ir),3),
    },
    "sleeve_summary": {
        "momentum_pnl": round(float(attr_df["mom_pnl"].sum()),4),
        "rate_pnl":     round(float(attr_df["rate_pnl"].sum()),4),
        "recon_pnl":    round(float(attr_df["recon_pnl"].sum()),4),
        "long_pnl":     round(float(attr_df["long_pnl"].sum()),4),
        "short_pnl":    round(float(attr_df["short_pnl"].sum()),4),
    },
    "drawdown_summary": {
        "n_drawdowns":       len(dd_df),
        "worst_drawdown":    round(float(dd_df["depth"].min()),4) if len(dd_df) > 0 else 0,
        "avg_recovery_days": round(float(dd_df["recovery_days"].dropna().mean()),1) if len(dd_df) > 0 else 0,
    },
    "distribution": {
        "skewness": round(float(ret.skew()),4),
        "kurtosis": round(float(ret.kurtosis()),4),
        "var_95":   round(var95,4),
        "cvar_95":  round(cvar95,4),
    }
}
with open(f"{RISK_DIR}/risk_report.json","w") as f:
    json.dump(risk_report, f, indent=2)

print(f"\n✓ Risk analysis complete — files saved to {RISK_DIR}")

Loading data...
Loaded.

Computing sleeve attribution...
  Momentum P&L:  129.14%
  Rate P&L:      14.04%
  Recon P&L:     0.00%
  Long book:     244.50%
  Short book:    -101.32%

Computing factor exposure...
  Market beta:      0.043
  Jensen's alpha:   2.34% p.a.
  R-squared:        0.004
  Info ratio:       0.000

Computing drawdowns...
  Drawdown periods: 32

  Top 5 worst drawdowns:
     start     trough   recovery   depth  duration_days  recovery_days
2020-03-27 2020-08-21 2021-08-16 -0.1822            147            360
2018-10-24 2019-03-12 2019-07-18 -0.1670            139            128
2022-04-01 2023-04-18 2024-05-15 -0.1504            382            393
2019-08-29 2020-01-16 2020-03-12 -0.1436            140             56
2015-09-08 2016-06-20 2016-11-08 -0.1150            286            141

Computing rolling metrics...
  Avg rolling Sharpe: 0.197
  Avg rolling vol:    11.22%
  Avg rolling beta:   0.053
  % time Sharpe > 1:  16.0%

Return distribution:
  Skewness:     -

In [54]:
import shutil
from google.colab import files
shutil.make_archive("/content/hedge_fund_risk", "zip",
                    "/content/hedge_fund/data/processed/risk")
files.download("/content/hedge_fund_risk.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [55]:
import pandas as pd
import numpy as np
import os, json

BASE         = "/content/hedge_fund/data/processed"
BACKTEST_DIR = f"{BASE}/backtest"
STRESS_DIR   = f"{BASE}/stress"
os.makedirs(STRESS_DIR, exist_ok=True)

# ── Load data ──────────────────────────────────────────────────────────────────
print("Loading data...")
nav     = pd.read_csv(f"{BACKTEST_DIR}/nav.csv",     index_col=0, parse_dates=True)
bt_ret  = pd.read_csv(f"{BACKTEST_DIR}/returns.csv", index_col=0, parse_dates=True)
weights = pd.read_csv(f"{BASE}/portfolio/weights_daily.csv", index_col=0, parse_dates=True)
returns = pd.read_csv(f"{BASE}/returns.csv",          index_col=0, parse_dates=True)
print("Loaded.")

ret    = bt_ret["return"].dropna()
strat  = nav["Strategy"]
bench  = nav["NIFTY500"]
RF     = 0.065 / 252

# ── Define stress periods ──────────────────────────────────────────────────────
STRESS_PERIODS = {
    "COVID Crash":            ("2020-02-19", "2020-03-23"),
    "COVID Recovery":         ("2020-03-24", "2020-12-31"),
    "Taper Tantrum India":    ("2018-08-01", "2018-10-31"),
    "IL&FS Crisis":           ("2018-09-20", "2019-03-31"),
    "US-China Trade War":     ("2019-05-01", "2019-08-31"),
    "RBI Rate Hike Cycle":    ("2022-05-04", "2023-02-08"),
    "Demonetisation Shock":   ("2016-11-08", "2016-12-31"),
    "Budget Selloff 2024":    ("2024-07-23", "2024-08-05"),
    "Russia-Ukraine Shock":   ("2022-02-24", "2022-03-15"),
    "Global Selloff 2015":    ("2015-08-18", "2015-09-30"),
}

# ── 1. Stress Period Analysis ──────────────────────────────────────────────────
print("\n── STRESS PERIOD ANALYSIS ──────────────────────────────────")
print(f"{'Event':<28} {'Strat':>8} {'NIFTY500':>10} {'Alpha':>8} {'MaxDD':>8} {'Beta':>7}")
print("─" * 75)

stress_results = {}
for name, (start, end) in STRESS_PERIODS.items():
    try:
        s_ret  = ret.loc[start:end]
        b_nav  = bench.loc[start:end]
        s_nav  = strat.loc[start:end]
        b_ret_s = b_nav.pct_change().dropna()

        if len(s_ret) < 3:
            continue

        total_s = float((1 + s_ret).prod() - 1)
        total_b = float((b_nav.iloc[-1] / b_nav.iloc[0]) - 1)
        alpha   = total_s - total_b
        max_dd  = float(((s_nav / s_nav.cummax()) - 1).min())

        # Beta during stress
        common = s_ret.index.intersection(b_ret_s.index)
        if len(common) > 5:
            cov  = float(np.cov(s_ret.loc[common], b_ret_s.loc[common])[0][1])
            var  = float(b_ret_s.loc[common].var())
            beta = cov / var if var > 0 else 0
        else:
            beta = 0

        stress_results[name] = {
            "start": start, "end": end,
            "strategy_return": round(total_s, 4),
            "benchmark_return": round(total_b, 4),
            "alpha": round(alpha, 4),
            "max_drawdown": round(max_dd, 4),
            "stress_beta": round(beta, 3),
            "n_days": len(s_ret),
        }

        flag = "✓" if total_s > total_b else "✗"
        print(f"{name:<28} {total_s:>7.1%} {total_b:>10.1%} {alpha:>8.1%} {max_dd:>7.1%} {beta:>7.2f}  {flag}")

    except Exception as e:
        print(f"{name:<28} ERROR: {e}")

# ── 2. Tail Risk Scenarios ─────────────────────────────────────────────────────
print(f"\n── TAIL RISK SCENARIOS ─────────────────────────────────────")

# Worst N days
worst_days = ret.nsmallest(10)
print(f"\n  10 Worst Single Days:")
for d, r in worst_days.items():
    b = bench.pct_change().loc[d] if d in bench.index else 0
    print(f"    {d.date()}  strat={r:.2%}  bench={float(b):.2%}")

# Worst N months
monthly = (1 + ret).resample("ME").prod() - 1
worst_months = monthly.nsmallest(5)
print(f"\n  5 Worst Months:")
for d, r in worst_months.items():
    print(f"    {d.strftime('%Y-%m')}  {r:.2%}")

# ── 3. Monte Carlo Simulation ──────────────────────────────────────────────────
print(f"\n── MONTE CARLO SIMULATION (1000 paths, 1 year) ─────────────")
np.random.seed(42)
N_SIMS    = 1000
N_DAYS    = 252
mu        = float(ret.mean())
sigma     = float(ret.std())
start_nav = 100.0

sim_ends  = []
sim_min   = []
for _ in range(N_SIMS):
    sim_ret = np.random.normal(mu, sigma, N_DAYS)
    sim_nav = start_nav * np.cumprod(1 + sim_ret)
    sim_ends.append(sim_nav[-1])
    sim_min.append(sim_nav.min())

sim_ends = np.array(sim_ends)
sim_min  = np.array(sim_min)

print(f"  Starting NAV:          ₹{start_nav:.0f}")
print(f"  Median ending NAV:     ₹{np.median(sim_ends):.2f}")
print(f"  10th percentile:       ₹{np.percentile(sim_ends, 10):.2f}")
print(f"  90th percentile:       ₹{np.percentile(sim_ends, 90):.2f}")
print(f"  Prob of loss (1yr):    {(sim_ends < start_nav).mean():.1%}")
print(f"  Prob of >10% gain:     {(sim_ends > start_nav*1.10).mean():.1%}")
print(f"  Expected worst drop:   {((sim_min - start_nav)/start_nav).mean():.1%}")
print(f"  5% worst-case drop:    {np.percentile((sim_min-start_nav)/start_nav, 5):.1%}")

# ── 4. Regime Analysis ────────────────────────────────────────────────────────
print(f"\n── PERFORMANCE BY MARKET REGIME ────────────────────────────")
bench_ret = bench.pct_change().dropna()
bench_ann = bench_ret.rolling(63).mean() * 252

regimes = pd.Series("NEUTRAL", index=bench_ret.index)
regimes[bench_ann > 0.15]  = "BULL"
regimes[bench_ann < -0.05] = "BEAR"

print(f"\n  {'Regime':<10} {'Days':>6} {'Strat CAGR':>12} {'Bench CAGR':>12} {'Alpha':>8}")
print("  " + "─" * 52)
for regime in ["BULL", "NEUTRAL", "BEAR"]:
    days = regimes[regimes == regime].index
    common = ret.index.intersection(days)
    if len(common) < 20: continue
    s_r = ret.loc[common]
    b_r = bench_ret.reindex(common).fillna(0)
    s_cagr = (1 + s_r.mean()) ** 252 - 1
    b_cagr = (1 + b_r.mean()) ** 252 - 1
    alpha  = s_cagr - b_cagr
    print(f"  {regime:<10} {len(common):>6} {s_cagr:>11.1%} {b_cagr:>12.1%} {alpha:>8.1%}")

# ── 5. Save results ────────────────────────────────────────────────────────────
pd.DataFrame(stress_results).T.to_csv(f"{STRESS_DIR}/stress_periods.csv")

mc_results = {
    "n_simulations":        N_SIMS,
    "horizon_days":         N_DAYS,
    "median_nav":           round(float(np.median(sim_ends)), 2),
    "p10_nav":              round(float(np.percentile(sim_ends, 10)), 2),
    "p90_nav":              round(float(np.percentile(sim_ends, 90)), 2),
    "prob_loss":            round(float((sim_ends < start_nav).mean()), 3),
    "prob_gain_10pct":      round(float((sim_ends > start_nav*1.10).mean()), 3),
    "expected_worst_drop":  round(float(((sim_min-start_nav)/start_nav).mean()), 3),
}
with open(f"{STRESS_DIR}/monte_carlo.json", "w") as f:
    json.dump(mc_results, f, indent=2)

with open(f"{STRESS_DIR}/stress_results.json", "w") as f:
    json.dump(stress_results, f, indent=2, default=str)

print(f"\n✓ Stress testing complete — saved to {STRESS_DIR}")

Loading data...
Loaded.

── STRESS PERIOD ANALYSIS ──────────────────────────────────
Event                           Strat   NIFTY500    Alpha    MaxDD    Beta
───────────────────────────────────────────────────────────────────────────
COVID Crash                    10.4%     -37.4%    47.8%   -2.2%    0.07  ✓
COVID Recovery                 -9.6%      81.0%   -90.6%  -18.2%    0.11  ✗
Taper Tantrum India             4.4%      -9.3%    13.7%   -4.0%   -0.29  ✓
IL&FS Crisis                   -9.2%       nan%     nan%  -16.7%   -0.24  ✗
US-China Trade War             20.9%       nan%     nan%   -4.5%   -0.09  ✗
RBI Rate Hike Cycle            -6.2%       4.4%   -10.6%  -11.6%   -0.13  ✗
Demonetisation Shock           -8.8%      -5.2%    -3.6%  -11.0%    0.64  ✗
Budget Selloff 2024            -2.8%      -1.4%    -1.4%   -3.0%    0.17  ✗
Russia-Ukraine Shock            5.9%       3.2%     2.7%   -1.5%   -0.28  ✓
Global Selloff 2015             3.0%      -6.5%     9.5%   -2.0%   -0.03  ✓

──

In [56]:
import shutil
from google.colab import files
shutil.make_archive("/content/hedge_fund_stress", "zip",
                    "/content/hedge_fund/data/processed/stress")
files.download("/content/hedge_fund_stress.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [57]:
import pandas as pd
import numpy as np
import json, os

BASE = "/content/hedge_fund/data/processed"

# ── Load all data ──────────────────────────────────────────────────────────────
nav    = pd.read_csv(f"{BASE}/backtest/nav.csv",     index_col=0, parse_dates=True)
bt_ret = pd.read_csv(f"{BASE}/backtest/returns.csv", index_col=0, parse_dates=True)
attr   = pd.read_csv(f"{BASE}/risk/sleeve_attribution_cumulative.csv", index_col=0, parse_dates=True)
dd     = pd.read_csv(f"{BASE}/risk/drawdown_series.csv", index_col=0, parse_dates=True)
roll   = pd.read_csv(f"{BASE}/risk/rolling_metrics.csv", index_col=0, parse_dates=True)

# Weekly resample for chart performance
nav_w  = nav.resample("W").last()
attr_w = attr.resample("W").last()
dd_w   = dd.resample("W").last()
roll_w = roll.resample("W").last()

ret         = bt_ret["return"].dropna()
annual      = (1 + ret).resample("YE").prod() - 1
bench_ann   = nav["NIFTY500"].pct_change().resample("YE").prod()
monthly     = (1 + ret).resample("ME").prod() - 1
monthly_df  = monthly.to_frame("ret")
monthly_df["year"]  = monthly_df.index.year
monthly_df["month"] = monthly_df.index.month

D = {
    "nav_dates":    nav_w.index.strftime("%Y-%m-%d").tolist(),
    "nav_strat":    nav_w["Strategy"].round(2).tolist(),
    "nav_nifty50":  nav_w["NIFTY50"].round(2).tolist(),
    "nav_nifty500": nav_w["NIFTY500"].round(2).tolist(),
    "dd_dates":     dd_w.index.strftime("%Y-%m-%d").tolist(),
    "dd_values":    (dd_w.iloc[:,0]*100).round(2).tolist(),
    "roll_dates":   roll_w.index.strftime("%Y-%m-%d").tolist(),
    "roll_sharpe":  roll_w["sharpe"].round(3).tolist(),
    "roll_vol":     (roll_w["vol"]*100).round(2).tolist(),
    "roll_beta":    roll_w["beta"].round(3).tolist(),
    "attr_dates":   attr_w.index.strftime("%Y-%m-%d").tolist(),
    "attr_mom":     (attr_w["mom_pnl"]*100).round(2).tolist(),
    "attr_rate":    (attr_w["rate_pnl"]*100).round(2).tolist(),
    "attr_recon":   (attr_w["recon_pnl"]*100).round(2).tolist(),
    "annual_years": [d.year for d in annual.index],
    "annual_strat": (annual*100).round(1).tolist(),
    "annual_bench": (bench_ann.reindex(annual.index).fillna(0)*100).round(1).tolist(),
    "monthly_data": [[int(r["year"]),int(r["month"]),round(float(r["ret"])*100,2)]
                     for _,r in monthly_df.iterrows()],
    "stats": {
        "cagr":       "8.8%",  "vol":     "11.3%",
        "sharpe":     "0.23",  "max_dd":  "-18.2%",
        "beta":       "0.043", "alpha":   "2.34%",
        "win_rate":   "49.9%", "ending_nav": "₹235.21",
        "total_ret":  "135.2%","sortino": "0.33",
        "calmar":     "0.48",  "pos_months": "57.6%",
    },
    "stress": {
        "COVID Crash":         [10.4, -37.4],
        "COVID Recovery":      [-9.6,  81.0],
        "Taper Tantrum":       [4.4,   -9.3],
        "RBI Hike Cycle":      [-6.2,   4.4],
        "Demonetisation":      [-8.8,  -5.2],
        "Russia-Ukraine":      [5.9,    3.2],
        "Global Selloff 2015": [3.0,   -6.5],
        "Budget Selloff 2024": [-2.8,  -1.4],
    }
}

data_json = json.dumps(D)

# ── Build HTML ────────────────────────────────────────────────────────────────
html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>India Multi-Strategy Hedge Fund — Performance Dashboard</title>
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.min.js"></script>
<link href="https://fonts.googleapis.com/css2?family=DM+Mono:wght@300;400;500&family=Fraunces:ital,opsz,wght@0,9..144,300;0,9..144,700;1,9..144,300&display=swap" rel="stylesheet">
<style>
  :root {{
    --bg:       #0a0c0f;
    --surface:  #111418;
    --border:   #1e2329;
    --text:     #e8eaed;
    --muted:    #6b7280;
    --gold:     #c9a84c;
    --gold2:    #e8c87a;
    --green:    #22c55e;
    --red:      #ef4444;
    --blue:     #3b82f6;
    --purple:   #a78bfa;
    --teal:     #2dd4bf;
  }}
  * {{ margin:0; padding:0; box-sizing:border-box; }}
  body {{
    background: var(--bg);
    color: var(--text);
    font-family: 'DM Mono', monospace;
    font-size: 13px;
    line-height: 1.6;
    min-height: 100vh;
  }}

  /* ── Header ── */
  .header {{
    padding: 48px 48px 32px;
    border-bottom: 1px solid var(--border);
    display: flex;
    justify-content: space-between;
    align-items: flex-end;
    background: linear-gradient(180deg, #0d1117 0%, var(--bg) 100%);
  }}
  .header-left h1 {{
    font-family: 'Fraunces', serif;
    font-size: 36px;
    font-weight: 300;
    color: var(--gold2);
    letter-spacing: -0.5px;
    line-height: 1.1;
  }}
  .header-left h1 em {{
    font-style: italic;
    color: var(--text);
  }}
  .header-left p {{
    color: var(--muted);
    margin-top: 8px;
    font-size: 11px;
    letter-spacing: 2px;
    text-transform: uppercase;
  }}
  .header-right {{
    text-align: right;
  }}
  .nav-badge {{
    font-family: 'Fraunces', serif;
    font-size: 48px;
    font-weight: 700;
    color: var(--gold);
    line-height: 1;
  }}
  .nav-label {{
    color: var(--muted);
    font-size: 10px;
    letter-spacing: 2px;
    text-transform: uppercase;
    margin-top: 4px;
  }}
  .period-tag {{
    display: inline-block;
    background: var(--border);
    color: var(--muted);
    font-size: 10px;
    padding: 3px 10px;
    border-radius: 2px;
    margin-top: 8px;
    letter-spacing: 1px;
  }}

  /* ── KPI Grid ── */
  .kpi-grid {{
    display: grid;
    grid-template-columns: repeat(6, 1fr);
    border-bottom: 1px solid var(--border);
  }}
  .kpi {{
    padding: 24px 28px;
    border-right: 1px solid var(--border);
    position: relative;
  }}
  .kpi:last-child {{ border-right: none; }}
  .kpi-label {{
    font-size: 10px;
    letter-spacing: 2px;
    text-transform: uppercase;
    color: var(--muted);
    margin-bottom: 8px;
  }}
  .kpi-value {{
    font-family: 'Fraunces', serif;
    font-size: 28px;
    font-weight: 700;
    line-height: 1;
  }}
  .kpi-value.pos {{ color: var(--green); }}
  .kpi-value.neg {{ color: var(--red); }}
  .kpi-value.gold {{ color: var(--gold); }}
  .kpi-value.neutral {{ color: var(--text); }}
  .kpi-sub {{
    font-size: 10px;
    color: var(--muted);
    margin-top: 6px;
  }}

  /* ── Main Grid ── */
  .main-grid {{
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 0;
  }}
  .chart-panel {{
    padding: 32px;
    border-right: 1px solid var(--border);
    border-bottom: 1px solid var(--border);
  }}
  .chart-panel.full {{
    grid-column: 1 / -1;
    border-right: none;
  }}
  .chart-panel.no-right {{ border-right: none; }}
  .panel-title {{
    font-size: 10px;
    letter-spacing: 3px;
    text-transform: uppercase;
    color: var(--gold);
    margin-bottom: 6px;
  }}
  .panel-subtitle {{
    font-size: 11px;
    color: var(--muted);
    margin-bottom: 24px;
  }}
  .chart-wrap {{
    position: relative;
    height: 220px;
  }}
  .chart-wrap.tall {{ height: 280px; }}

  /* ── Annual Bar Chart ── */
  .annual-bars {{
    display: flex;
    align-items: flex-end;
    gap: 6px;
    height: 180px;
    padding-top: 20px;
    margin-top: 8px;
  }}
  .bar-group {{
    flex: 1;
    display: flex;
    flex-direction: column;
    align-items: center;
    gap: 3px;
    height: 100%;
    justify-content: flex-end;
  }}
  .bar-pair {{
    display: flex;
    gap: 2px;
    align-items: flex-end;
    width: 100%;
  }}
  .bar {{
    flex: 1;
    min-height: 2px;
    border-radius: 1px 1px 0 0;
    transition: opacity 0.2s;
    cursor: pointer;
    position: relative;
  }}
  .bar:hover {{ opacity: 0.8; }}
  .bar-strat {{ background: var(--gold); }}
  .bar-bench {{ background: var(--border); border: 1px solid #2a3040; }}
  .bar-neg {{ border-radius: 0 0 1px 1px; }}
  .bar-year {{
    font-size: 9px;
    color: var(--muted);
    letter-spacing: 0;
    margin-top: 6px;
    transform: rotate(-45deg);
    transform-origin: center;
  }}
  .bar-val {{
    font-size: 9px;
    color: var(--muted);
    text-align: center;
    min-height: 14px;
  }}

  /* ── Monthly Heatmap ── */
  .heatmap {{
    margin-top: 8px;
  }}
  .heatmap-grid {{
    display: grid;
    grid-template-columns: 40px repeat(12, 1fr);
    gap: 2px;
  }}
  .hm-header {{
    font-size: 9px;
    color: var(--muted);
    text-align: center;
    padding: 4px 0;
    letter-spacing: 1px;
  }}
  .hm-year {{
    font-size: 9px;
    color: var(--muted);
    display: flex;
    align-items: center;
    padding-right: 8px;
  }}
  .hm-cell {{
    height: 28px;
    border-radius: 2px;
    display: flex;
    align-items: center;
    justify-content: center;
    font-size: 8px;
    font-weight: 500;
    cursor: pointer;
    transition: transform 0.1s;
  }}
  .hm-cell:hover {{ transform: scale(1.1); z-index: 10; position: relative; }}

  /* ── Stress Table ── */
  .stress-grid {{
    margin-top: 8px;
    display: grid;
    gap: 6px;
  }}
  .stress-row {{
    display: grid;
    grid-template-columns: 180px 1fr 1fr 80px;
    align-items: center;
    gap: 12px;
    padding: 10px 14px;
    background: var(--surface);
    border-radius: 3px;
    border: 1px solid var(--border);
  }}
  .stress-name {{
    font-size: 11px;
    color: var(--text);
  }}
  .stress-bar-wrap {{
    height: 8px;
    background: var(--border);
    border-radius: 2px;
    overflow: hidden;
    position: relative;
  }}
  .stress-bar-fill {{
    height: 100%;
    border-radius: 2px;
    position: absolute;
    top: 0;
  }}
  .stress-val {{
    font-family: 'Fraunces', serif;
    font-size: 14px;
    font-weight: 700;
    text-align: right;
  }}
  .stress-alpha {{
    font-size: 10px;
    color: var(--muted);
    text-align: right;
  }}

  /* ── Legend ── */
  .legend {{
    display: flex;
    gap: 20px;
    margin-bottom: 16px;
    flex-wrap: wrap;
  }}
  .legend-item {{
    display: flex;
    align-items: center;
    gap: 6px;
    font-size: 10px;
    color: var(--muted);
    letter-spacing: 1px;
  }}
  .legend-dot {{
    width: 8px;
    height: 8px;
    border-radius: 1px;
    flex-shrink: 0;
  }}

  /* ── Footer ── */
  .footer {{
    padding: 24px 48px;
    border-top: 1px solid var(--border);
    display: flex;
    justify-content: space-between;
    color: var(--muted);
    font-size: 10px;
    letter-spacing: 1px;
  }}

  /* ── Tooltip ── */
  .tooltip {{
    position: fixed;
    background: var(--surface);
    border: 1px solid var(--border);
    padding: 8px 12px;
    border-radius: 3px;
    font-size: 11px;
    pointer-events: none;
    display: none;
    z-index: 100;
    color: var(--text);
  }}
</style>
</head>
<body>
<div class="tooltip" id="tooltip"></div>

<!-- Header -->
<div class="header">
  <div class="header-left">
    <h1>India <em>Multi-Strategy</em><br>Hedge Fund</h1>
    <p>Performance &amp; Risk Dashboard — 2015–2024</p>
  </div>
  <div class="header-right">
    <div class="nav-badge" id="ending-nav">₹235.21</div>
    <div class="nav-label">Ending NAV per ₹100 invested</div>
    <div class="period-tag">JAN 2015 — DEC 2024 · 10 YEARS</div>
  </div>
</div>

<!-- KPI Grid -->
<div class="kpi-grid">
  <div class="kpi">
    <div class="kpi-label">CAGR</div>
    <div class="kpi-value pos">8.8%</div>
    <div class="kpi-sub">10-year annualised</div>
  </div>
  <div class="kpi">
    <div class="kpi-label">Sharpe Ratio</div>
    <div class="kpi-value gold">0.23</div>
    <div class="kpi-sub">vs 6.5% risk-free</div>
  </div>
  <div class="kpi">
    <div class="kpi-label">Max Drawdown</div>
    <div class="kpi-value neg">-18.2%</div>
    <div class="kpi-sub">Mar–Aug 2020</div>
  </div>
  <div class="kpi">
    <div class="kpi-label">Market Beta</div>
    <div class="kpi-value gold">0.043</div>
    <div class="kpi-sub">Near market-neutral</div>
  </div>
  <div class="kpi">
    <div class="kpi-label">Jensen's Alpha</div>
    <div class="kpi-value pos">+2.34%</div>
    <div class="kpi-sub">Annual excess return</div>
  </div>
  <div class="kpi">
    <div class="kpi-label">COVID Alpha</div>
    <div class="kpi-value pos">+47.8%</div>
    <div class="kpi-sub">vs NIFTY500 crash</div>
  </div>
</div>

<!-- Row 1: NAV + Drawdown -->
<div class="main-grid">
  <div class="chart-panel full">
    <div class="panel-title">Equity Curve</div>
    <div class="panel-subtitle">NAV rebased to ₹100 — Strategy vs NIFTY50 vs NIFTY500</div>
    <div class="legend">
      <div class="legend-item"><div class="legend-dot" style="background:var(--gold)"></div>Strategy</div>
      <div class="legend-item"><div class="legend-dot" style="background:var(--blue)"></div>NIFTY50</div>
      <div class="legend-item"><div class="legend-dot" style="background:var(--purple)"></div>NIFTY500</div>
    </div>
    <div class="chart-wrap tall"><canvas id="navChart"></canvas></div>
  </div>

  <div class="chart-panel">
    <div class="panel-title">Drawdown</div>
    <div class="panel-subtitle">Underwater curve from peak NAV</div>
    <div class="chart-wrap"><canvas id="ddChart"></canvas></div>
  </div>

  <div class="chart-panel no-right">
    <div class="panel-title">Rolling 12-Month Sharpe</div>
    <div class="panel-subtitle">1-year rolling window, annualised</div>
    <div class="chart-wrap"><canvas id="sharpeChart"></canvas></div>
  </div>

  <div class="chart-panel">
    <div class="panel-title">Sleeve Attribution</div>
    <div class="panel-subtitle">Cumulative P&L contribution by sleeve</div>
    <div class="legend">
      <div class="legend-item"><div class="legend-dot" style="background:var(--gold)"></div>Momentum</div>
      <div class="legend-item"><div class="legend-dot" style="background:var(--teal)"></div>Rate</div>
      <div class="legend-item"><div class="legend-dot" style="background:var(--purple)"></div>Recon</div>
    </div>
    <div class="chart-wrap"><canvas id="attrChart"></canvas></div>
  </div>

  <div class="chart-panel no-right">
    <div class="panel-title">Annual Returns</div>
    <div class="panel-subtitle">Strategy (gold) vs NIFTY500 (grey)</div>
    <div id="annualBars"></div>
  </div>

  <!-- Stress Tests -->
  <div class="chart-panel full">
    <div class="panel-title">Stress Period Analysis</div>
    <div class="panel-subtitle">Strategy return vs NIFTY500 during major market events</div>
    <div class="stress-grid" id="stressGrid"></div>
  </div>

  <!-- Monthly Heatmap -->
  <div class="chart-panel full" style="border-bottom:none">
    <div class="panel-title">Monthly Returns Heatmap</div>
    <div class="panel-subtitle">Green = positive · Red = negative · Intensity = magnitude</div>
    <div class="heatmap" id="heatmap"></div>
  </div>
</div>

<!-- Footer -->
<div class="footer">
  <span>INDIA MULTI-STRATEGY L/S EQUITY — INTERNAL USE ONLY — NOT FOR DISTRIBUTION</span>
  <span>BACKTEST 2015–2024 · PAST PERFORMANCE DOES NOT GUARANTEE FUTURE RESULTS</span>
</div>

<script>
const D = {data_json};

Chart.defaults.color = '#6b7280';
Chart.defaults.borderColor = '#1e2329';
Chart.defaults.font.family = "'DM Mono', monospace";
Chart.defaults.font.size = 10;

const pluginNoDataLabel = {{
  id: 'noData',
  afterDraw(chart) {{
    const {{ ctx, data, chartArea }} = chart;
    // nothing needed
  }}
}};

// Shared chart options
const baseOptions = (yLabel='') => ({{
  responsive: true,
  maintainAspectRatio: false,
  interaction: {{ mode:'index', intersect:false }},
  plugins: {{
    legend: {{ display: false }},
    tooltip: {{
      backgroundColor: '#111418',
      borderColor: '#1e2329',
      borderWidth: 1,
      titleColor: '#6b7280',
      bodyColor: '#e8eaed',
      padding: 10,
      callbacks: {{
        title: items => items[0].label,
      }}
    }}
  }},
  scales: {{
    x: {{
      grid: {{ color: '#1a1f27', drawTicks: false }},
      ticks: {{ maxTicksLimit: 8, color: '#4b5563' }},
    }},
    y: {{
      grid: {{ color: '#1a1f27', drawTicks: false }},
      ticks: {{ color: '#4b5563' }},
      title: {{ display: !!yLabel, text: yLabel, color: '#4b5563' }}
    }}
  }}
}});

// ── NAV Chart ────────────────────────────────────────────────────────────────
new Chart(document.getElementById('navChart'), {{
  type: 'line',
  data: {{
    labels: D.nav_dates,
    datasets: [
      {{
        label: 'Strategy',
        data: D.nav_strat,
        borderColor: '#c9a84c',
        backgroundColor: 'rgba(201,168,76,0.06)',
        borderWidth: 2,
        pointRadius: 0,
        fill: true,
        tension: 0.3,
      }},
      {{
        label: 'NIFTY50',
        data: D.nav_nifty50,
        borderColor: '#3b82f6',
        backgroundColor: 'transparent',
        borderWidth: 1.5,
        pointRadius: 0,
        borderDash: [4,4],
        tension: 0.3,
      }},
      {{
        label: 'NIFTY500',
        data: D.nav_nifty500,
        borderColor: '#a78bfa',
        backgroundColor: 'transparent',
        borderWidth: 1.5,
        pointRadius: 0,
        borderDash: [2,4],
        tension: 0.3,
      }},
    ]
  }},
  options: {{
    ...baseOptions('₹ NAV'),
    plugins: {{
      ...baseOptions().plugins,
      tooltip: {{
        ...baseOptions().plugins.tooltip,
        callbacks: {{
          title: items => items[0].label,
          label: item => ` ${{item.dataset.label}}: ₹${{item.raw?.toFixed(2)}}`,
        }}
      }}
    }}
  }}
}});

// ── Drawdown Chart ────────────────────────────────────────────────────────────
new Chart(document.getElementById('ddChart'), {{
  type: 'line',
  data: {{
    labels: D.dd_dates,
    datasets: [{{
      label: 'Drawdown',
      data: D.dd_values,
      borderColor: '#ef4444',
      backgroundColor: 'rgba(239,68,68,0.12)',
      borderWidth: 1.5,
      pointRadius: 0,
      fill: true,
      tension: 0.2,
    }}]
  }},
  options: {{
    ...baseOptions('%'),
    plugins: {{
      ...baseOptions().plugins,
      tooltip: {{
        ...baseOptions().plugins.tooltip,
        callbacks: {{
          title: i => i[0].label,
          label: i => ` Drawdown: ${{i.raw?.toFixed(1)}}%`,
        }}
      }}
    }}
  }}
}});

// ── Sharpe Chart ──────────────────────────────────────────────────────────────
const sharpeData = D.roll_sharpe.map(v => isNaN(v) ? null : v);
new Chart(document.getElementById('sharpeChart'), {{
  type: 'line',
  data: {{
    labels: D.roll_dates,
    datasets: [
      {{
        label: 'Rolling Sharpe',
        data: sharpeData,
        borderColor: '#2dd4bf',
        backgroundColor: 'rgba(45,212,191,0.06)',
        borderWidth: 1.5,
        pointRadius: 0,
        fill: true,
        tension: 0.3,
        spanGaps: true,
      }},
      {{
        label: 'Sharpe = 1',
        data: D.roll_dates.map(() => 1),
        borderColor: 'rgba(107,114,128,0.4)',
        borderWidth: 1,
        borderDash: [4,4],
        pointRadius: 0,
        fill: false,
      }}
    ]
  }},
  options: baseOptions('Sharpe'),
}});

// ── Attribution Chart ─────────────────────────────────────────────────────────
new Chart(document.getElementById('attrChart'), {{
  type: 'line',
  data: {{
    labels: D.attr_dates,
    datasets: [
      {{
        label: 'Momentum',
        data: D.attr_mom,
        borderColor: '#c9a84c',
        backgroundColor: 'transparent',
        borderWidth: 2,
        pointRadius: 0,
        tension: 0.3,
      }},
      {{
        label: 'Rate',
        data: D.attr_rate,
        borderColor: '#2dd4bf',
        backgroundColor: 'transparent',
        borderWidth: 1.5,
        pointRadius: 0,
        tension: 0.3,
      }},
      {{
        label: 'Recon',
        data: D.attr_recon,
        borderColor: '#a78bfa',
        backgroundColor: 'transparent',
        borderWidth: 1.5,
        pointRadius: 0,
        tension: 0.3,
      }},
    ]
  }},
  options: {{
    ...baseOptions('%'),
    plugins: {{
      ...baseOptions().plugins,
      tooltip: {{
        ...baseOptions().plugins.tooltip,
        callbacks: {{
          title: i => i[0].label,
          label: i => ` ${{i.dataset.label}}: ${{i.raw?.toFixed(1)}}%`,
        }}
      }}
    }}
  }}
}});

// ── Annual Bars ───────────────────────────────────────────────────────────────
(function() {{
  const container = document.getElementById('annualBars');
  const years   = D.annual_years;
  const strat   = D.annual_strat;
  const bench   = D.annual_bench;
  const maxAbs  = Math.max(...strat.map(Math.abs), ...bench.map(Math.abs));
  const H       = 160;

  let html = '<div class="annual-bars">';
  for (let i = 0; i < years.length; i++) {{
    const s = strat[i], b = bench[i];
    const sH = Math.round(Math.abs(s) / maxAbs * H * 0.85);
    const bH = Math.round(Math.abs(b) / maxAbs * H * 0.85);
    const sCol = s >= 0 ? '#c9a84c' : '#ef4444';
    const bCol = b >= 0 ? '#2a3040' : '#7f1d1d';
    const sign = s >= 0 ? '+' : '';
    html += `
    <div class="bar-group">
      <div class="bar-val" style="color:${{sCol}}">${{sign}}${{s}}%</div>
      <div class="bar-pair">
        <div class="bar ${{s<0?'bar-neg':''}}" style="height:${{sH}}px;background:${{sCol}}"></div>
        <div class="bar ${{b<0?'bar-neg':''}}" style="height:${{bH}}px;background:${{bCol}}"></div>
      </div>
      <div class="bar-year">${{years[i]}}</div>
    </div>`;
  }}
  html += '</div>';
  container.innerHTML = html;
}})();

// ── Stress Grid ───────────────────────────────────────────────────────────────
(function() {{
  const grid = document.getElementById('stressGrid');
  const stress = D.stress;
  const maxAbs = Math.max(...Object.values(stress).map(([s,b]) => Math.max(Math.abs(s),Math.abs(b))));
  let html = '';
  for (const [name, [s, b]] of Object.entries(stress)) {{
    const alpha = s - b;
    const sW = Math.round(Math.abs(s) / maxAbs * 100);
    const bW = Math.round(Math.abs(b) / maxAbs * 100);
    const sCol = s >= 0 ? '#22c55e' : '#ef4444';
    const bCol = b >= 0 ? '#3b82f6' : '#dc2626';
    const aCol = alpha >= 0 ? '#22c55e' : '#ef4444';
    html += `
    <div class="stress-row">
      <div class="stress-name">${{name}}</div>
      <div>
        <div style="font-size:9px;color:var(--muted);margin-bottom:3px">STRATEGY</div>
        <div class="stress-bar-wrap">
          <div class="stress-bar-fill" style="width:${{sW}}%;background:${{sCol}};${{s<0?'right:0':'left:0'}}"></div>
        </div>
      </div>
      <div>
        <div style="font-size:9px;color:var(--muted);margin-bottom:3px">NIFTY500</div>
        <div class="stress-bar-wrap">
          <div class="stress-bar-fill" style="width:${{bW}}%;background:${{bCol}};${{b<0?'right:0':'left:0'}}"></div>
        </div>
      </div>
      <div>
        <div class="stress-val" style="color:${{sCol}}">${{s>=0?'+':''}}${{s}}%</div>
        <div class="stress-alpha" style="color:${{aCol}}">α ${{alpha>=0?'+':''}}${{alpha.toFixed(1)}}%</div>
      </div>
    </div>`;
  }}
  grid.innerHTML = html;
}})();

// ── Monthly Heatmap ───────────────────────────────────────────────────────────
(function() {{
  const months = ['J','F','M','A','M','J','J','A','S','O','N','D'];
  const monthsFull = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'];
  const data = {{}};
  D.monthly_data.forEach(([y,m,r]) => {{
    if (!data[y]) data[y] = {{}};
    data[y][m] = r;
  }});
  const years = Object.keys(data).sort();
  const allVals = D.monthly_data.map(([,, r]) => r);
  const maxAbs = Math.max(...allVals.map(Math.abs));

  function cellColor(v) {{
    if (v === undefined) return 'var(--surface)';
    const intensity = Math.min(Math.abs(v) / maxAbs, 1);
    if (v > 0) {{
      const g = Math.round(80 + intensity * 120);
      const r = Math.round(10 + intensity * 20);
      return `rgb(${{r}},${{g}},50)`;
    }} else {{
      const r = Math.round(100 + intensity * 140);
      const g = Math.round(10 + intensity * 20);
      return `rgb(${{r}},${{g}},30)`;
    }}
  }}

  const tt = document.getElementById('tooltip');
  let html = '<div class="heatmap-grid">';
  html += '<div class="hm-header"></div>';
  months.forEach(m => html += `<div class="hm-header">${{m}}</div>`);
  years.forEach(yr => {{
    html += `<div class="hm-year">${{yr}}</div>`;
    for (let m = 1; m <= 12; m++) {{
      const v = data[yr]?.[m];
      const col = cellColor(v);
      const label = v !== undefined ? `${{yr}} ${{monthsFull[m-1]}}: ${{v>=0?'+':''}}${{v?.toFixed(1)}}%` : '';
      const textCol = v !== undefined && Math.abs(v) > maxAbs * 0.3 ? '#fff' : 'rgba(255,255,255,0.4)';
      html += `<div class="hm-cell" style="background:${{col}};color:${{textCol}}"
        onmouseenter="showTT(event,'${{label}}')" onmouseleave="hideTT()">${{v!==undefined?v.toFixed(1):''}}</div>`;
    }}
  }});
  html += '</div>';
  document.getElementById('heatmap').innerHTML = html;
}})();

function showTT(e, text) {{
  const tt = document.getElementById('tooltip');
  tt.textContent = text;
  tt.style.display = 'block';
  tt.style.left = (e.clientX + 12) + 'px';
  tt.style.top  = (e.clientY - 28) + 'px';
}}
function hideTT() {{
  document.getElementById('tooltip').style.display = 'none';
}}
</script>
</body>
</html>"""

# Save and download
path = "/content/hedge_fund_dashboard.html"
with open(path, "w") as f:
    f.write(html)

from google.colab import files
files.download(path)
print(f"✓ Dashboard saved — {len(html):,} bytes")
print("Open hedge_fund_dashboard.html in any browser")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Dashboard saved — 89,142 bytes
Open hedge_fund_dashboard.html in any browser
